# Generative Causal AI for Public Health Policy Optimization
## Bangladesh Demographic & Health Survey 2011–2022

| Layer | Method | Key Output |
|-------|--------|-----------|
| 1 | Multi-wave cluster linkage | Trajectory matrix 2011–2022 |
| 2 | Climate-shock fusion (CSI) | CLIMATE_SHOCK_IDX per cluster |
| 3 | NOTEARS on cluster-wave panel | 8 division DAGs + SHD |
| 4a | MissForest imputation | Complete panel + 2022 dataset |
| 4b | Tabular VAE + k-means (mixed loss) | Vulnerability profiles |
| 5 | CTGAN / TVAE | Synthetic records |
| 5b | LightGBM + SHAP (cluster split) | Feature attribution |
| 6 | PPO (all 8 divisions) | Optimal intervention sequences |
| 7 | NSGA-II Pareto (5 runs) | Equity–efficiency frontier |


In [ ]:
pass


## Section 1 — Install Dependencies


In [ ]:
import subprocess, sys

def pip_install(pkg):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg], capture_output=True, text=True)
    if r.returncode != 0:
        for l in [l for l in r.stderr.strip().splitlines() if l.strip()][-3:]:
            print(f'     {l}')
for pkg in ['pyreadstat', 'gcastle', 'torch', 'ctgan', 'sdmetrics', 'stable-baselines3[extra]', 'gymnasium', 'pymoo', 'shap', 'lightgbm', 'rapidfuzz', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn', 'plotly', 'networkx', 'pyarrow', 'lingam', 'cdt']:
    pip_install(pkg)


## Section 2 — Dataset Paths


In [ ]:
import os
from pathlib import Path
DATASET_ROOT = Path('')
_r = DATASET_ROOT
FILE_MAP = {'BD_2011': {'ir': str(_r / 'BD_2011_DHS_05232026_1412_236689' / 'BDIR61SV' / 'BDIR61FL.SAV'), 'kr': str(_r / 'BD_2011_DHS_05232026_1412_236689' / 'BDKR61SV' / 'BDKR61FL.SAV'), 'gc': str(_r / 'BD_2011_DHS_05232026_1412_236689' / 'BDGC62FL' / 'BDGC62FL.csv'), 'year': 2011}, 'BD_2014': {'ir': str(_r / 'BD_2014_DHS_05232026_1412_236689' / 'BDIR72SV' / 'BDIR72FL.SAV'), 'kr': str(_r / 'BD_2014_DHS_05232026_1412_236689' / 'BDKR72SV' / 'BDKR72FL.SAV'), 'gc': str(_r / 'BD_2014_DHS_05232026_1412_236689' / 'BDGC72FL' / 'BDGC72FL.csv'), 'year': 2014}, 'BD_2017': {'ir': str(_r / 'BD_2017-18_DHS_05232026_1411_236689' / 'BDIR7RSV' / 'BDIR7RFL.SAV'), 'kr': str(_r / 'BD_2017-18_DHS_05232026_1411_236689' / 'BDKR7RSV' / 'BDKR7RFL.SAV'), 'gc': str(_r / 'BD_2017-18_DHS_05232026_1411_236689' / 'BDGC7RFL' / 'BDGC7RFL' / 'BDGC7RFL.csv'), 'year': 2017}, 'BD_2022': {'ir': str(_r / 'BD_2022_DHS_05232026_149_236689' / 'BDIR81SV' / 'BDIR81FL.SAV'), 'kr': str(_r / 'BD_2022_DHS_05232026_149_236689' / 'BDKR81SV' / 'BDKR81FL.SAV'), 'gc': str(_r / 'BD_2022_DHS_05232026_149_236689' / 'BDGC81FL' / 'BDGC81FL.csv'), 'year': 2022}}
OUTPUT_DIR = Path('/kaggle/working/bdhs_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_ok = True
for wk, meta in FILE_MAP.items():
    print(f'  {wk} ({meta['year']}):')
    for key in ['ir', 'kr', 'gc']:
        p = Path(meta[key])
        if p.exists():
            pass
        else:
            all_ok = False
if not all_ok:
    for root, dirs, files in os.walk('/kaggle/input/datasets/ssaha108'):
        for f in files:
            if f.endswith('.SAV') or (f.endswith('.csv') and 'GC' in f.upper()):
                pass
else:
    pass


## Section 3 — Imports & Config


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import lightgbm as lgb
import shap
import networkx as nx
import json, pickle, os
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa', 'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'DejaVu Sans', 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('husl')
DIVISIONS = {1: 'Barisal', 2: 'Chittagong', 3: 'Dhaka', 4: 'Khulna', 5: 'Rajshahi', 6: 'Sylhet', 7: 'Rangpur', 8: 'Mymensingh'}
IR_VARS_COMMON = ['V001', 'V002', 'V003', 'V005', 'V006', 'V007', 'V012', 'V024', 'V025', 'V106', 'V133', 'V113', 'V116', 'V136', 'V149', 'V169A', 'V171A', 'V190', 'V467D', 'V743A', 'V743B', 'V743D', 'V501', 'V160']
KR_VARS = ['V001', 'V002', 'V003', 'B8', 'B19', 'HW1', 'HW13', 'HW70', 'HW71', 'HW72', 'M14', 'M15', 'V024', 'V025', 'V190', 'V005']
GC_VARS = ['DHSCLUST', 'Precipitation_2015', 'Precipitation_2020', 'Rainfall_2015', 'Rainfall_2020', 'Wet_Days_2015', 'Wet_Days_2020', 'Drought_Episodes', 'Travel_Times', 'Mean_Temperature_2015', 'Mean_Temperature_2020', 'Elevation', 'Enhanced_Vegetation_Index_2015', 'Enhanced_Vegetation_Index_2020', 'UN_Population_Density_2015', 'UN_Population_Density_2020']


## Section 4 — Data Loading


In [ ]:
def load_sav_subset(path, usecols):
    try:
        df, _ = pyreadstat.read_sav(path, usecols=usecols, apply_value_formats=False)
        missing = [c for c in usecols if c not in df.columns]
        if missing:
            pass
        return df
    except Exception as e:
        return pd.DataFrame()

def improved_toilet_flag(v116, year, v160=None):
    """
    DHS-8 p.700-701: era-specific JMP toilet classification.

    SDG era (surveys ≥2022):
      Improved facility types: V116 ∈ {11,12,13,15,21,22,41}
      AND not shared: V160 ≠ 1  (DHS-8 p.700: shared facility = unimproved under SDG)
      V160=0 → not shared (improved); V160=1 → shared (unimproved); NaN → treat as improved
        11=flush/piped sewer  12=flush/septic  13=flush/pit
        15=flush/don't know where (IMPROVED — not same as 14)
        21=VIP  22=pit with slab  41=composting
        NOTE: 14=flush/somewhere else = UNIMPROVED in all eras

    MDG era (surveys 2011–2017):
      Shared facilities were unimproved; code 14 still unimproved.
      adds 23=pit latrine without slab, 31=other slab facility
    """
    v = pd.to_numeric(v116, errors='coerce')
    if year >= 2022:
        improved_type = v.isin([11, 12, 13, 15, 21, 22, 41])
        if v160 is not None:
            v160_num = pd.to_numeric(v160, errors='coerce')
            not_shared = (v160_num != 1).fillna(True)
            return (improved_type & not_shared).astype(float)
        return improved_type.astype(float)
    return v.isin([11, 12, 13, 15, 21, 22, 23, 31, 41]).astype(float)

def load_wave(wave_key):
    meta = FILE_MAP[wave_key]
    year = meta['year']
    ir = load_sav_subset(meta['ir'], IR_VARS_COMMON)
    if ir.empty or 'V001' not in ir.columns:
        return {'ir': pd.DataFrame(), 'kr': pd.DataFrame(), 'gc': pd.DataFrame(), 'year': year}
    ir['WAVE'] = year
    ir['CLUSTER_ID'] = ir['V001'].astype(float).astype(int)
    for col in ['V169A', 'V171A']:
        if col not in ir.columns:
            ir[col] = np.nan
    _v160 = ir['V160'] if 'V160' in ir.columns else None
    ir['IMPROVED_TOILET_FLAG'] = improved_toilet_flag(ir['V116'], year, _v160) if 'V116' in ir.columns else np.nan
    dec_cols = [c for c in ['V743A', 'V743B', 'V743D'] if c in ir.columns]
    if dec_cols:
        dec_mask = ir['V501'] == 1 if 'V501' in ir.columns else pd.Series(True, index=ir.index)
        ir['DECISION_AUTONOMY'] = np.where(dec_mask, ir[dec_cols].isin([1, 2]).sum(axis=1) / len(dec_cols), np.nan)
    else:
        ir['DECISION_AUTONOMY'] = np.nan
    print(f'  IR: {len(ir):,} women')
    kr = load_sav_subset(meta['kr'], KR_VARS)
    if not kr.empty and 'V001' in kr.columns:
        kr['WAVE'] = year
        kr['CLUSTER_ID'] = kr['V001'].astype(float).astype(int)
        for z in ['HW70', 'HW71', 'HW72']:
            if z in kr.columns:
                kr[z] = pd.to_numeric(kr[z], errors='coerce') / 100.0
                kr.loc[kr[z].abs() >= 6.0, z] = np.nan
        if 'HW13' in kr.columns:
            kr = kr[kr['HW13'].isin([0]) | kr['HW13'].isna()].copy()
        if 'M14' in kr.columns:
            kr['M14'] = pd.to_numeric(kr['M14'], errors='coerce')
            kr.loc[kr['M14'] >= 96, 'M14'] = np.nan
        if 'M15' in kr.columns:
            kr['INST_DELIVERY'] = kr['M15'].between(20, 49).astype(float)
            kr.loc[kr['M15'].isna(), 'INST_DELIVERY'] = np.nan
        print(f'  KR: {len(kr):,} children')
    else:
        kr = pd.DataFrame()
    gc = pd.read_csv(meta['gc'], low_memory=False)
    print(f'  GC cols: {gc.columns.tolist()[:8]}...')
    gc_avail = [c for c in GC_VARS if c in gc.columns]
    gc = gc[gc_avail].copy()
    if 'DHSCLUST' in gc.columns:
        gc = gc.rename(columns={'DHSCLUST': 'CLUSTER_ID'})
    for _tt in ['Travel_Times', 'Travel_Times_2015', 'Travel_Times_2000', 'Travel_time_to_city']:
        if _tt in gc.columns and 'Travel_Times' not in gc.columns:
            gc['Travel_Times'] = gc[_tt]
            break
    prec_found = False
    for suf in ['2020', '2015', '2010', '2005', '2000']:
        for raw in [f'Precipitation_{suf}', f'Annual_Precipitation_{suf}', f'Mean_Precipitation_{suf}', f'Total_Precipitation_{suf}', f'Rainfall_{suf}', f'Annual_Rainfall_{suf}']:
            if raw in gc.columns and (not prec_found):
                gc['PRECIPITATION'] = gc[raw]
                prec_found = True
    if not prec_found:
        all_cols = pd.read_csv(meta['gc'], nrows=0).columns.tolist()
        for c in all_cols:
            if any((kw in c.lower() for kw in ['precipit', 'rainfall', 'rain'])):
                gc['PRECIPITATION'] = pd.read_csv(meta['gc'], usecols=[c], low_memory=False)[c]
                prec_found = True
                break
    if not prec_found:
        gc['PRECIPITATION'] = 0.0
    gc['WAVE'] = year
    z_prec = stats.zscore(gc['PRECIPITATION'].fillna(gc['PRECIPITATION'].median()), nan_policy='omit')
    z_drought = stats.zscore(gc['Drought_Episodes'].fillna(gc['Drought_Episodes'].median()), nan_policy='omit') if 'Drought_Episodes' in gc.columns else np.zeros(len(gc))
    gc['CLIMATE_SHOCK_IDX'] = (np.array(z_prec) + np.array(z_drought)) / 2
    print(f'  GC: {len(gc):,} clusters')
    return {'ir': ir, 'kr': kr, 'gc': gc, 'year': year}
waves = {}
for wk in ['BD_2011', 'BD_2014', 'BD_2017', 'BD_2022']:
    waves[wk] = load_wave(wk)


## Section 5 — Layer 1: Multi-Wave Cluster Panel


In [ ]:
def build_cluster_features(wd):
    ir, kr, gc, year = (wd['ir'], wd['kr'], wd['gc'], wd['year'])
    ir_num = ir.select_dtypes(include=[np.number]).copy()
    ir_num['CLUSTER_ID'] = ir['CLUSTER_ID']
    ir_num['DIVISION'] = ir['V024'] if 'V024' in ir.columns else np.nan
    ir_num['URBAN'] = (ir['V025'] == 1).astype(float) if 'V025' in ir.columns else np.nan
    ir_num['EDU_YRS'] = ir['V133'].clip(0, 25) if 'V133' in ir.columns else np.nan
    ir_num['WEALTH_Q'] = ir['V190'].clip(1, 5) if 'V190' in ir.columns else np.nan
    ir_num['MOBILE_OWN'] = (ir['V169A'] == 1).astype(float) if 'V169A' in ir.columns else np.nan
    ir_num['INTERNET'] = (ir['V171A'] == 1).astype(float) if 'V171A' in ir.columns else np.nan
    _w_imp = [11, 12, 13, 14, 21, 31, 41, 51, 61, 62, 71] if year >= 2018 else [11, 12, 13, 14, 21, 31, 41, 51, 71]
    ir_num['IMPROVED_WATER'] = ir['V113'].isin(_w_imp).astype(float) if 'V113' in ir.columns else np.nan
    ir_num['IMPROVED_TOILET'] = ir['IMPROVED_TOILET_FLAG'] if 'IMPROVED_TOILET_FLAG' in ir.columns else np.nan
    ir_num['HH_SIZE'] = ir['V136'].clip(1, 30) if 'V136' in ir.columns else np.nan
    ir_num['DIST_HEALTH'] = (ir['V467D'] == 1).astype(float) if 'V467D' in ir.columns else np.nan
    clust = ir_num.groupby('CLUSTER_ID').agg(DIVISION=('DIVISION', 'first'), URBAN=('URBAN', 'first'), EDU_YRS=('EDU_YRS', 'mean'), WEALTH_Q=('WEALTH_Q', 'mean'), MOBILE_OWN=('MOBILE_OWN', 'mean'), INTERNET=('INTERNET', 'mean'), IMPROVED_WATER=('IMPROVED_WATER', 'mean'), IMPROVED_TOILET=('IMPROVED_TOILET', 'mean'), HH_SIZE=('HH_SIZE', 'mean'), DIST_HEALTH=('DIST_HEALTH', 'mean'), N_WOMEN=('CLUSTER_ID', 'count')).reset_index()
    if isinstance(kr, pd.DataFrame) and (not kr.empty) and ('CLUSTER_ID' in kr.columns):
        if 'B19' in kr.columns:
            kr = kr[kr['B19'].between(0, 59)].copy()
        elif 'B8' in kr.columns:
            kr = kr[kr['B8'] <= 4].copy()
        kr_agg = {}
        if 'HW70' in kr.columns:
            kr_agg['STUNTING'] = ('HW70', lambda x: (x < -2).mean())
        if 'M14' in kr.columns:
            kr_agg['ANC_4PLUS'] = ('M14', lambda x: (x[x < 96] >= 4).mean())
        if 'INST_DELIVERY' in kr.columns:
            kr_agg['INST_DELIV'] = ('INST_DELIVERY', 'mean')
        if kr_agg:
            clust_kr = kr.groupby('CLUSTER_ID').agg(**kr_agg).reset_index()
            clust = clust.merge(clust_kr, on='CLUSTER_ID', how='left')
    gc_cols = ['CLUSTER_ID', 'CLIMATE_SHOCK_IDX'] + [c for c in ['PRECIPITATION', 'Travel_Times'] if c in gc.columns]
    clust = clust.merge(gc[gc_cols].drop_duplicates('CLUSTER_ID'), on='CLUSTER_ID', how='left')
    clust['WAVE'] = year
    parts = []
    if 'DIST_HEALTH' in clust.columns:
        parts.append(clust['DIST_HEALTH'].fillna(0))
    if 'ANC_4PLUS' in clust.columns:
        parts.append(1 - clust['ANC_4PLUS'].fillna(0))
    if 'INST_DELIV' in clust.columns:
        parts.append(1 - clust['INST_DELIV'].fillna(0))
    if parts:
        clust['HEALTHCARE_EXCLUSION'] = np.mean(parts, axis=0)
    return clust
cluster_dfs = {}
for wk, wd in waves.items():
    cluster_dfs[wk] = build_cluster_features(wd)
    print(f'  {wk}: {len(cluster_dfs[wk])} clusters')
all_clusters = pd.concat(cluster_dfs.values(), ignore_index=True)
cluster_counts = all_clusters.groupby('CLUSTER_ID')['WAVE'].nunique()
persistent_4 = cluster_counts[cluster_counts == 4].index
print(f'\nPersistent clusters (all 4 waves): {len(persistent_4):,}')
linkage_summary = pd.DataFrame({'Wave Pair': ['2011-2014', '2014-2017', '2017-2022', 'All four waves'], 'Clusters Linked': [len(set(cluster_dfs['BD_2011']['CLUSTER_ID']) & set(cluster_dfs['BD_2014']['CLUSTER_ID'])), len(set(cluster_dfs['BD_2014']['CLUSTER_ID']) & set(cluster_dfs['BD_2017']['CLUSTER_ID'])), len(set(cluster_dfs['BD_2017']['CLUSTER_ID']) & set(cluster_dfs['BD_2022']['CLUSTER_ID'])), len(persistent_4)]})
linkage_summary['Match Rate (%)'] = (linkage_summary['Clusters Linked'] / cluster_counts.shape[0] * 100).round(1)
print(linkage_summary.to_string(index=False))
linkage_summary.to_csv(OUTPUT_DIR / 'table1_linkage.csv', index=False)
traj_cols = [c for c in ['CLUSTER_ID', 'WAVE', 'DIVISION', 'STUNTING', 'HEALTHCARE_EXCLUSION', 'ANC_4PLUS', 'INST_DELIV', 'EDU_YRS', 'WEALTH_Q', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'DIST_HEALTH', 'CLIMATE_SHOCK_IDX'] if c in all_clusters.columns]
trajectory_df = all_clusters[all_clusters['CLUSTER_ID'].isin(persistent_4)][traj_cols].copy()
trajectory_df.to_parquet(OUTPUT_DIR / 'trajectory_matrix.parquet')
if 'STUNTING' in trajectory_df.columns:
    trend = trajectory_df.groupby(['WAVE', 'DIVISION'])['STUNTING'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(12, 5))
    for dc, dn in DIVISIONS.items():
        sub = trend[trend['DIVISION'] == dc]
        if len(sub) > 1:
            ax.plot(sub['WAVE'], sub['STUNTING'] * 100, marker='o', label=dn, linewidth=2)
    ax.set(xlabel='Survey Wave', ylabel='Stunting Prevalence (%)', title='Child Stunting Prevalence by Division (BDHS 2011-2022)')
    ax.set_xticks([2011, 2014, 2017, 2022])
    ax.legend(bbox_to_anchor=(1.02, 1), fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig1_stunting_trend.png', dpi=150, bbox_inches='tight')
    plt.show()


## Section 6 — Layer 2: Climate Fusion & Analytical Dataset


In [ ]:
def build_analytical_dataset(wd):
    ir, kr, gc, year = (wd['ir'].copy(), wd['kr'], wd['gc'].copy(), wd['year'])
    if ir.empty or 'V001' not in ir.columns:
        return pd.DataFrame()
    ir['EDU_YRS'] = pd.to_numeric(ir.get('V133', np.nan), errors='coerce').clip(0, 25)
    ir['WEALTH_Q'] = pd.to_numeric(ir.get('V190', np.nan), errors='coerce').clip(1, 5)
    ir['MOBILE_OWN'] = (ir['V169A'] == 1).astype(float) if 'V169A' in ir.columns else np.nan
    ir['INTERNET'] = (ir['V171A'] == 1).astype(float) if 'V171A' in ir.columns else np.nan
    _w_imp = [11, 12, 13, 14, 21, 31, 41, 51, 61, 62, 71] if year >= 2018 else [11, 12, 13, 14, 21, 31, 41, 51, 71]
    ir['IMPROVED_WATER'] = ir['V113'].isin(_w_imp).astype(float) if 'V113' in ir.columns else np.nan
    ir['IMPROVED_TOILET'] = ir['IMPROVED_TOILET_FLAG'] if 'IMPROVED_TOILET_FLAG' in ir.columns else np.nan
    ir['HH_SIZE'] = pd.to_numeric(ir.get('V136', np.nan), errors='coerce').clip(1, 30)
    if 'V467D' in ir.columns:
        _v467d = ir['V467D'].copy()
        _v467d[_v467d == 3] = np.nan
        ir['DIST_HEALTH'] = (_v467d == 1).astype(float).where(_v467d.notna())
    else:
        ir['DIST_HEALTH'] = np.nan
    ir['URBAN'] = (ir['V025'] == 1).astype(float) if 'V025' in ir.columns else np.nan
    ir['DIVISION'] = ir['V024'].astype(float) if 'V024' in ir.columns else np.nan
    ir['SURVEY_WEIGHT'] = pd.to_numeric(ir.get('V005', np.nan), errors='coerce') / 1000000
    if isinstance(kr, pd.DataFrame) and (not kr.empty):
        agg = {}
        if 'HW70' in kr.columns:
            if 'B19' in kr.columns:
                kr_6 = kr[kr['B19'].between(0, 59)].copy()
            elif 'B8' in kr.columns:
                kr_6 = kr[kr['B8'] <= 4].copy()
            else:
                kr_6 = kr

            def _wtd_stunt(grp):
                w = kr_6.loc[grp.index, 'V005'].fillna(1) / 1000000 if 'V005' in kr_6.columns else np.ones(len(grp))
                valid = grp.notna()
                return np.average(grp[valid] < -2, weights=w[valid]) if valid.any() else np.nan
            agg['STUNTING'] = kr_6.groupby('V001')['HW70'].apply(_wtd_stunt)
        if 'M14' in kr.columns:

            def _wtd_anc(grp):
                w = kr.loc[grp.index, 'V005'].fillna(1) / 1000000 if 'V005' in kr.columns else np.ones(len(grp))
                valid = (grp < 96) & grp.notna()
                return np.average(grp[valid] >= 4, weights=w[valid]) if valid.any() else np.nan
            agg['ANC_4PLUS'] = kr.groupby('V001')['M14'].apply(_wtd_anc)
        if 'INST_DELIVERY' in kr.columns:

            def _wtd_inst(grp):
                w = kr.loc[grp.index, 'V005'].fillna(1) / 1000000 if 'V005' in kr.columns else np.ones(len(grp))
                valid = grp.notna()
                return np.average(grp[valid], weights=w[valid]) if valid.any() else np.nan
            agg['INST_DELIV'] = kr.groupby('V001')['INST_DELIVERY'].apply(_wtd_inst)
        if agg:
            kr_c = pd.DataFrame(agg).reset_index().rename(columns={'V001': 'CLUSTER_ID'})
            ir = ir.merge(kr_c, left_on='V001', right_on='CLUSTER_ID', how='left')
    gc_keep = ['CLUSTER_ID', 'CLIMATE_SHOCK_IDX'] + [c for c in ['PRECIPITATION', 'Travel_Times'] if c in gc.columns]
    ir = ir.merge(gc[gc_keep].drop_duplicates('CLUSTER_ID'), left_on='V001', right_on='CLUSTER_ID', how='left')
    ir['WAVE'] = year
    return ir
analytical_dfs = {}
for wk, wd in waves.items():
    analytical_dfs[wk] = build_analytical_dataset(wd)
    print(f'  {wk}: {len(analytical_dfs[wk]):,} women')
ANALYSIS_2022 = analytical_dfs['BD_2022']
if ANALYSIS_2022.empty:
    raise RuntimeError('BD_2022 IR failed.')
CAUSAL_FEATURES = ['CLIMATE_SHOCK_IDX', 'URBAN', 'EDU_YRS', 'WEALTH_Q', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'HH_SIZE', 'DIST_HEALTH', 'ANC_4PLUS', 'INST_DELIV', 'STUNTING', 'DIVISION']
causal_cols = [c for c in CAUSAL_FEATURES if c in ANALYSIS_2022.columns]
CAUSAL_DF = ANALYSIS_2022[causal_cols].dropna(subset=['STUNTING']).copy()
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey=True)
axes = axes.flatten()
for i, (dc, dn) in enumerate(DIVISIONS.items()):
    sub = CAUSAL_DF[CAUSAL_DF['DIVISION'] == dc]['CLIMATE_SHOCK_IDX']
    if len(sub) > 10:
        axes[i].hist(sub.dropna(), bins=30, color='#2196F3', alpha=0.7, edgecolor='white')
        axes[i].set_title(dn, fontsize=10, fontweight='bold')
        axes[i].axvline(sub.median(), color='red', linestyle='--', linewidth=1.5)
    else:
        axes[i].set_visible(False)
plt.suptitle('CSI Distribution by Division (BDHS 2022)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_csi.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 7 — Layer 3: Division-Stratified Causal Discovery


In [ ]:
CAUSAL_NODE_COLS = ['CLIMATE_SHOCK_IDX', 'URBAN', 'EDU_YRS', 'WEALTH_Q', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'HH_SIZE', 'DIST_HEALTH', 'ANC_4PLUS', 'INST_DELIV', 'STUNTING']

def build_cluster_panel(df):
    cols = CAUSAL_NODE_COLS + ['CLUSTER_ID', 'WAVE', 'DIVISION']
    avail = [c for c in cols if c in df.columns]
    return df[avail].dropna(subset=['STUNTING']).copy()

def run_notears_division(df_div, name):
    node_cols = [c for c in CAUSAL_NODE_COLS if c in df_div.columns]
    X = df_div[node_cols].copy()
    Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=node_cols)
    try:
        import os as _os, logging
        _os.environ['CASTLE_BACKEND'] = 'pytorch'
        logging.disable(logging.CRITICAL)
        from castle.algorithms import Notears
        m = Notears(w_threshold=0.01)
        m.learn(Xs.values)
        adj = pd.DataFrame(m.causal_matrix, index=node_cols, columns=node_cols)
        logging.disable(logging.NOTSET)
        method = 'NOTEARS'
    except Exception as e:
        corr = Xs.corr().abs()
        np.fill_diagonal(corr.values, 0)
        adj = pd.DataFrame(np.triu((corr > 0.15).astype(float).values, k=1), index=node_cols, columns=node_cols)
        method = 'CorrHeuristic'
    n_edges = int((adj.abs() > 0.01).sum().sum())
    csi_anc = float(adj.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS']) if 'CLIMATE_SHOCK_IDX' in node_cols and 'ANC_4PLUS' in node_cols else 0.0
    csi_stun = float(adj.loc['CLIMATE_SHOCK_IDX', 'STUNTING']) if 'CLIMATE_SHOCK_IDX' in node_cols and 'STUNTING' in node_cols else 0.0
    return {'division': name, 'adj': adj, 'node_cols': node_cols, 'n_edges': n_edges, 'n_obs': len(Xs), 'method': method, 'csi_to_anc4plus': csi_anc, 'csi_to_stunting': csi_stun}

def structural_hamming_distance(adj1, adj2, cols):
    c = [x for x in cols if x in adj1.index and x in adj2.index]
    a1 = (adj1.loc[c, c].values > 0.01).astype(int)
    a2 = (adj2.loc[c, c].values > 0.01).astype(int)
    sk1, sk2 = (np.maximum(a1, a1.T), np.maximum(a2, a2.T))
    skel_diff = int(np.sum(sk1 != sk2))
    sk_agree = (sk1 == 1) & (sk2 == 1)
    n = len(c)
    reversals = sum((1 for i in range(n) for j in range(i + 1, n) if sk_agree[i, j] and a1[i, j] != a2[i, j]))
    return skel_diff + reversals
_cache_path = OUTPUT_DIR / 'cluster_panel_imputed.parquet'
if _cache_path.exists():
    cluster_panel = pd.read_parquet(_cache_path)
    CLUSTER_PANEL_IMPUTED = cluster_panel.copy()
else:
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

    def _missforest_impute_panel(df, max_iter=3, label=''):
        df_imp = df.copy()
        for col in df_imp.columns:
            if df_imp[col].isna().any():
                df_imp[col] = df_imp[col].fillna(df_imp[col].median())
        for it in range(max_iter):
            for col in df_imp.columns:
                miss = df[col].isna()
                if miss.sum() == 0:
                    continue
                feats = [c for c in df_imp.columns if c != col]
                X_tr = df_imp.loc[~miss, feats]
                y_tr = df_imp.loc[~miss, col]
                X_pr = df_imp.loc[miss, feats]
                if len(y_tr) < 10 or len(X_pr) == 0:
                    continue
                rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42) if y_tr.nunique() <= 5 else RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)
                try:
                    rf.fit(X_tr, y_tr)
                    df_imp.loc[miss, col] = rf.predict(X_pr)
                except:
                    pass
        return df_imp
    raw_panel = build_cluster_panel(all_clusters)
    panel_cols = [c for c in CAUSAL_NODE_COLS if c in raw_panel.columns]
    meta_cols = [c for c in ['CLUSTER_ID', 'WAVE', 'DIVISION'] if c in raw_panel.columns]
    print(f'Raw panel: {raw_panel.shape}. Imputing cluster-wave panel (MissForest, inline)...')
    panel_imp = _missforest_impute_panel(raw_panel[panel_cols].copy(), max_iter=3, label='Panel')
    CLUSTER_PANEL_IMPUTED = raw_panel[meta_cols].copy()
    for c in panel_cols:
        CLUSTER_PANEL_IMPUTED[c] = panel_imp[c].values
    print(f'Panel nulls post-imputation: {CLUSTER_PANEL_IMPUTED[panel_cols].isna().sum().sum()}')
    CLUSTER_PANEL_IMPUTED.to_parquet(_cache_path)
    cluster_panel = CLUSTER_PANEL_IMPUTED.copy()
print(f'Panel ready: {cluster_panel.shape}  (rows = cluster-wave obs)')
div_results = {}
for dc, dn in DIVISIONS.items():
    sub = cluster_panel[cluster_panel['DIVISION'] == dc].copy()
    node_avail = [c for c in CAUSAL_NODE_COLS if c in sub.columns]
    sub_clean = sub[node_avail].dropna()
    if len(sub_clean) < 30:
        continue
    print(f'  {dn}: n={len(sub_clean)}...')
    r = run_notears_division(sub_clean, dn)
    div_results[dc] = r
    print(f'    edges={r['n_edges']}  CSI→ANC4+={r['csi_to_anc4plus']:.3f}  ({r['method']})')
print('\n  National (pooled)...')
nat_clean = cluster_panel[[c for c in CAUSAL_NODE_COLS if c in cluster_panel.columns]].dropna()
national_result = run_notears_division(nat_clean, 'National')
div_results[0] = national_result
div_codes_done = [k for k in div_results if k > 0]
n_divs = len(div_codes_done)
all_node_cols = [c for c in CAUSAL_NODE_COLS if c in cluster_panel.columns]
shd_matrix = np.zeros((n_divs, n_divs), dtype=int)
for i, di in enumerate(div_codes_done):
    for j, dj in enumerate(div_codes_done):
        if i != j:
            shd_matrix[i, j] = structural_hamming_distance(div_results[di]['adj'], div_results[dj]['adj'], all_node_cols)
shd_df = pd.DataFrame(shd_matrix, index=[DIVISIONS[k] for k in div_codes_done], columns=[DIVISIONS[k] for k in div_codes_done])
mean_shd = shd_matrix[shd_matrix > 0].mean() if shd_matrix.any() else 0
table2 = pd.DataFrame([{'Division': r['division'], 'N obs': r['n_obs'], 'Edges': r['n_edges'], 'CSI→ANC4+ path': 'Present' if r['csi_to_anc4plus'] > 0.01 else 'Absent', 'CSI→ANC4+ w': f'{r['csi_to_anc4plus']:.3f}' if r['csi_to_anc4plus'] > 0.01 else '—', 'Method': r['method']} for dc, r in div_results.items() if dc > 0])
print(table2.to_string(index=False))
print(f'Mean SHD: {mean_shd:.1f}')
table2.to_csv(OUTPUT_DIR / 'table2_causal_discovery.csv', index=False)
conv_log = {DIVISIONS.get(k, 'National'): {'method': div_results[k]['method'], 'converged': div_results[k]['method'] == 'NOTEARS', 'n_obs': div_results[k]['n_obs'], 'csi_anc4plus_w': div_results[k]['csi_to_anc4plus']} for k in div_results}
with open(OUTPUT_DIR / 'notears_convergence_log.json', 'w') as f:
    json.dump(conv_log, f, indent=2)
_pathway_divs = {dc: res for dc, res in div_results.items() if dc > 0 and res.get('csi_to_anc4plus', 0) > 0.01}
if _pathway_divs:
    import matplotlib.pyplot as _plt
    for _dc, _res in _pathway_divs.items():
        _dname = DIVISIONS.get(_dc, f'Div{_dc}')
        _sub = (CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel)[(CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel)['DIVISION'] == float(_dc)]
        if len(_sub) < 20:
            continue
        _boot_ws = []
        np.random.seed(42)
        for _b in range(500):
            _samp = _sub.sample(n=len(_sub), replace=True)
            try:
                from castle.algorithms import Notears as _NT
                import logging
                logging.disable(logging.CRITICAL)
                _sc = StandardScaler()
                _xs = _sc.fit_transform(_samp[CAUSAL_NODE_COLS].dropna())
                _m = _NT(w_threshold=0.01)
                _m.learn(_xs)
                _adj = pd.DataFrame(_m.causal_matrix, index=CAUSAL_NODE_COLS, columns=CAUSAL_NODE_COLS)
                logging.disable(logging.NOTSET)
                _ci = CAUSAL_NODE_COLS.index('CLIMATE_SHOCK_IDX')
                _ai = CAUSAL_NODE_COLS.index('ANC_4PLUS')
                _boot_ws.append(float(_adj.iloc[_ci, _ai]))
            except Exception:
                _boot_ws.append(0.0)
        _arr = np.array(_boot_ws)
        _pct = float((_arr > 0.01).mean() * 100)
        _mean = float(_arr.mean())
        _sd = float(_arr.std())
        _lo = float(np.percentile(_arr, 2.5))
        _hi = float(np.percentile(_arr, 97.5))
        print(f'  {_dname}: mean={_mean:.3f} SD={_sd:.3f} 95%CI=[{_lo:.3f},{_hi:.3f}] positive={_pct:.1f}%')
        _fig, _ax = _plt.subplots(figsize=(7, 3))
        _ax.hist(_arr, bins=40, color='steelblue', alpha=0.7)
        _ax.axvline(_mean, color='red', lw=2, label=f'Mean={_mean:.3f}')
        _ax.axvline(_lo, color='gray', lw=1.2, ls='--', label='2.5/97.5 pct')
        _ax.axvline(_hi, color='gray', lw=1.2, ls='--')
        _ax.set_xlabel('CSI→ANC4+ edge weight')
        _ax.set_ylabel('Resamples')
        _ax.set_title(f'Bootstrap: {_dname} CSI→ANC4+ (B=500, positive={_pct:.1f}%)')
        _ax.legend(fontsize=8)
        _plt.tight_layout()
        _plt.savefig(OUTPUT_DIR / f'fig_bootstrap_{_dname.lower()}.png', dpi=150, bbox_inches='tight')
        _plt.show()
        conv_log[_dname]['bootstrap_mean'] = round(_mean, 4)
        conv_log[_dname]['bootstrap_sd'] = round(_sd, 4)
        conv_log[_dname]['bootstrap_95ci_lo'] = round(_lo, 4)
        conv_log[_dname]['bootstrap_95ci_hi'] = round(_hi, 4)
        conv_log[_dname]['bootstrap_pct_pos'] = round(_pct, 2)
    with open(OUTPUT_DIR / 'notears_convergence_log.json', 'w') as _f:
        json.dump(conv_log, _f, indent=2)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if n_divs > 1:
    sns.heatmap(shd_df, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0], linewidths=0.5, cbar_kws={'label': 'SHD'})
    axes[0].set_title(f'Pairwise SHD (Mean={mean_shd:.1f})', fontsize=12, fontweight='bold')
G = nx.DiGraph()
G.add_nodes_from(all_node_cols)
adj_nat = national_result['adj']
for src in all_node_cols:
    for tgt in all_node_cols:
        w = adj_nat.loc[src, tgt] if src in adj_nat.index and tgt in adj_nat.columns else 0
        if abs(w) > 0.01:
            G.add_edge(src, tgt, weight=float(w))
pos = nx.spring_layout(G, seed=42, k=2.5)
nc = ['#FF5722' if n == 'CLIMATE_SHOCK_IDX' else '#F44336' if n == 'STUNTING' else '#2196F3' for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, ax=axes[1], node_color=nc, node_size=800, alpha=0.9)
nx.draw_networkx_labels(G, pos, ax=axes[1], font_size=7, labels={n: n.replace('_', '\n') for n in G.nodes()})
nx.draw_networkx_edges(G, pos, ax=axes[1], width=[G[u][v]['weight'] * 3 for u, v in G.edges()], edge_color='#555', arrows=True, arrowsize=15, connectionstyle='arc3,rad=0.1')
axes[1].set_title('National Causal DAG', fontsize=12, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_dag.png', dpi=150, bbox_inches='tight')
plt.show()

def _run_notears_raw(df_div, name, scale=True):
    """Run NOTEARS with or without z-standardisation."""
    node_cols = [c for c in CAUSAL_NODE_COLS if c in df_div.columns]
    X = df_div[node_cols].dropna().copy()
    if scale:
        Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=node_cols)
    else:
        Xs = X.copy()
    try:
        import logging as _lg
        _lg.disable(_lg.CRITICAL)
        from castle.algorithms import Notears
        m = Notears(w_threshold=0.01)
        m.learn(Xs.values)
        adj = pd.DataFrame(m.causal_matrix, index=node_cols, columns=node_cols)
        _lg.disable(_lg.NOTSET)
    except Exception as e:
        return None
    return (adj.abs() > 0.01).astype(int)
zscore_vs_raw_results = []
for dc, dn in DIVISIONS.items():
    sub = cluster_panel[cluster_panel['DIVISION'] == dc].copy()
    sub_clean = sub[[c for c in CAUSAL_NODE_COLS if c in sub.columns]].dropna()
    if len(sub_clean) < 30:
        continue
    adj_z = _run_notears_raw(sub_clean, dn, scale=True)
    adj_raw = _run_notears_raw(sub_clean, dn, scale=False)
    if adj_z is None or adj_raw is None:
        continue
    diff_edges = int((adj_z.values != adj_raw.values).sum())
    total_edges = adj_z.size
    csi_anc_z = int(adj_z.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS']) if 'CLIMATE_SHOCK_IDX' in adj_z.index else 0
    csi_anc_raw = int(adj_raw.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS']) if 'CLIMATE_SHOCK_IDX' in adj_raw.index else 0
    zscore_vs_raw_results.append({'Division': dn, 'n_obs': len(sub_clean), 'Edges_zscored': int(adj_z.values.sum()), 'Edges_raw': int(adj_raw.values.sum()), 'Changed_edges': diff_edges, 'Pct_changed': round(diff_edges / total_edges * 100, 1), 'CSI_ANC_zscored': csi_anc_z, 'CSI_ANC_raw': csi_anc_raw, 'Orientation_changed': csi_anc_z != csi_anc_raw})
    print(f'  {dn}: z-scored_edges={adj_z.values.sum()} raw_edges={adj_raw.values.sum()} changed={diff_edges}/{total_edges} ({diff_edges / total_edges * 100:.1f}%) CSI→ANC4+: z={csi_anc_z} raw={csi_anc_raw} orientation_changed={csi_anc_z != csi_anc_raw}')
zscore_raw_df = pd.DataFrame(zscore_vs_raw_results)
print('\nSummary: z-scored vs. raw NOTEARS comparison')
print(zscore_raw_df.to_string(index=False))
zscore_raw_df.to_csv(OUTPUT_DIR / 'table_zscore_vs_raw_notears.csv', index=False)
with open(OUTPUT_DIR / 'zscore_vs_raw_notears.json', 'w') as f:
    json.dump(zscore_vs_raw_results, f, indent=2)
LAMBDA_GRID = [0.05, 0.1, 0.2]
reg_stability_results = []
for dc, dn in DIVISIONS.items():
    sub = cluster_panel[cluster_panel['DIVISION'] == dc].copy()
    sub_clean = sub[[c for c in CAUSAL_NODE_COLS if c in sub.columns]].dropna()
    if len(sub_clean) < 30:
        continue
    node_cols = [c for c in CAUSAL_NODE_COLS if c in sub_clean.columns]
    Xs = pd.DataFrame(StandardScaler().fit_transform(sub_clean[node_cols]), columns=node_cols)
    adj_per_lambda = {}
    for lam in LAMBDA_GRID:
        try:
            import logging as _lg
            _lg.disable(_lg.CRITICAL)
            from castle.algorithms import Notears
            m = Notears(lambda1=lam, w_threshold=0.01)
            m.learn(Xs.values)
            adj_per_lambda[lam] = pd.DataFrame(m.causal_matrix, index=node_cols, columns=node_cols)
            _lg.disable(_lg.NOTSET)
        except Exception as e:
            adj_per_lambda[lam] = None
    valid_adjs = [a for a in adj_per_lambda.values() if a is not None]
    if len(valid_adjs) < 2:
        continue
    stacked = np.stack([(a.abs() > 0.01).astype(int).values for a in valid_adjs])
    stable_mask = (stacked.sum(axis=0) == 0) | (stacked.sum(axis=0) == len(valid_adjs))
    n_stable = int(stable_mask.sum())
    n_total = stable_mask.size
    pct_stable = round(n_stable / n_total * 100, 1)
    csi_anc_per_lam = {}
    for lam, adj in adj_per_lambda.items():
        if adj is not None and 'CLIMATE_SHOCK_IDX' in adj.index and ('ANC_4PLUS' in adj.columns):
            csi_anc_per_lam[lam] = int(adj.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS'] > 0.01)
    csi_anc_consistent = len(set(csi_anc_per_lam.values())) == 1
    reg_stability_results.append({'Division': dn, 'n_obs': len(sub_clean), 'Stable_edges_pct': pct_stable, 'N_stable': n_stable, 'N_total': n_total, 'CSI_ANC_lam005': csi_anc_per_lam.get(0.05, 'NA'), 'CSI_ANC_lam010': csi_anc_per_lam.get(0.1, 'NA'), 'CSI_ANC_lam020': csi_anc_per_lam.get(0.2, 'NA'), 'CSI_ANC_consistent_across_lambda': csi_anc_consistent})
    print(f'  {dn}: stable_edges={pct_stable}%  CSI→ANC4+: λ={csi_anc_per_lam}  consistent={csi_anc_consistent}')
reg_stability_df = pd.DataFrame(reg_stability_results)
print('\nRegularisation grid stability summary:')
print(reg_stability_df.to_string(index=False))
reg_stability_df.to_csv(OUTPUT_DIR / 'table_lambda_grid_stability.csv', index=False)
with open(OUTPUT_DIR / 'lambda_grid_stability.json', 'w') as f:
    json.dump(reg_stability_results, f, indent=2)
NEGATIVE_CONTROLS = [('STUNTING', 'CLIMATE_SHOCK_IDX', 'Stunting→CSI (impossible: outcome cannot cause weather)'), ('ANC_4PLUS', 'CLIMATE_SHOCK_IDX', 'ANC4+→CSI (impossible: healthcare use cannot cause rainfall)'), ('IMPROVED_TOILET', 'WEALTH_Q', 'Sanitation→Wealth (sanitation is downstream of wealth, not upstream)'), ('INTERNET', 'EDU_YRS', 'Internet→Education_years (internet access follows education, not precedes it)')]
neg_ctrl_results = []
print(f'Testing {len(NEGATIVE_CONTROLS)} negative-control (scientifically implausible) edges:')
for dc, dn in DIVISIONS.items():
    sub = cluster_panel[cluster_panel['DIVISION'] == dc].copy()
    sub_clean = sub[[c for c in CAUSAL_NODE_COLS if c in sub.columns]].dropna()
    if len(sub_clean) < 30:
        continue
    res_div = div_results.get(dc)
    if res_div is None:
        continue
    adj = res_div['adj']
    for src, tgt, label in NEGATIVE_CONTROLS:
        if src in adj.index and tgt in adj.columns:
            w = float(adj.loc[src, tgt])
            detected = w > 0.01
            neg_ctrl_results.append({'Division': dn, 'Control_edge': f'{src}→{tgt}', 'Label': label, 'Weight': round(w, 4), 'Detected': detected})
neg_ctrl_df = pd.DataFrame(neg_ctrl_results)
n_detected = neg_ctrl_df['Detected'].sum()
n_total_ctrl = len(neg_ctrl_df)
fpr = n_detected / n_total_ctrl if n_total_ctrl > 0 else 0
print(f'\nNegative controls recovered by NOTEARS: {n_detected}/{n_total_ctrl} ({fpr:.1%})')
print(neg_ctrl_df.groupby('Control_edge')['Detected'].sum().reset_index().to_string(index=False))
neg_ctrl_df.to_csv(OUTPUT_DIR / 'table_negative_controls.csv', index=False)
with open(OUTPUT_DIR / 'negative_control_results.json', 'w') as f:
    json.dump(neg_ctrl_results, f, indent=2, default=str)

def run_golem_division(df_div, name):
    """GOLEM: relaxes equal-variance assumption of NOTEARS.
    FIX v13: gcastle Golem API varies by version — try multiple signatures.
    equal_variances=False (EV-GOLEM) is the scale-relaxed variant requested
    by reviewers (Lachapelle et al. 2019).
    """
    node_cols = [c for c in CAUSAL_NODE_COLS if c in df_div.columns]
    X = df_div[node_cols].dropna()
    Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=node_cols)
    import logging as _lg
    _lg.disable(_lg.CRITICAL)
    adj = None
    tried = []
    for kwargs in [dict(num_iter=int(10000.0), equal_variances=False, lambda_1=0.002, lambda_2=5.0), dict(num_iter=int(10000.0), equal_variances=False), dict(num_iter=int(10000.0)), dict()]:
        try:
            from castle.algorithms import GOLEM
            m = GOLEM(**kwargs)
            m.learn(Xs.values)
            adj = pd.DataFrame(m.causal_matrix, index=node_cols, columns=node_cols)
            tried.append(f'success with kwargs={list(kwargs.keys())}')
            break
        except Exception as e:
            tried.append(f'failed({e})')
    _lg.disable(_lg.NOTSET)
    if adj is None:
        return None
    csi_anc = float(adj.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS']) if 'CLIMATE_SHOCK_IDX' in node_cols and 'ANC_4PLUS' in node_cols else 0.0
    return {'division': name, 'adj': adj, 'node_cols': node_cols, 'n_edges': int((adj.abs() > 0.01).sum().sum()), 'csi_to_anc4plus': csi_anc, 'method': 'GOLEM'}

def run_lingam_division(df_div, name):
    """LiNGAM: non-Gaussian noise, structurally identifiable DAG."""
    node_cols = [c for c in CAUSAL_NODE_COLS if c in df_div.columns]
    X = df_div[node_cols].dropna()
    Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=node_cols)
    try:
        import lingam
        m = lingam.DirectLiNGAM(random_state=42)
        m.fit(Xs.values)
        adj = pd.DataFrame(m.adjacency_matrix_.T, index=node_cols, columns=node_cols)
        csi_anc = float(adj.loc['CLIMATE_SHOCK_IDX', 'ANC_4PLUS']) if 'CLIMATE_SHOCK_IDX' in node_cols and 'ANC_4PLUS' in node_cols else 0.0
        return {'division': name, 'adj': adj, 'node_cols': node_cols, 'n_edges': int((adj.abs() > 0.01).sum().sum()), 'csi_to_anc4plus': csi_anc, 'method': 'LiNGAM'}
    except Exception as e:
        return None
golem_results = {}
lingam_results = {}
consensus_rows = []
for dc, dn in list(DIVISIONS.items()) + [(0, 'National')]:
    if dc == 0:
        sub_clean = cluster_panel[[c for c in CAUSAL_NODE_COLS if c in cluster_panel.columns]].dropna()
    else:
        sub = cluster_panel[cluster_panel['DIVISION'] == dc].copy()
        sub_clean = sub[[c for c in CAUSAL_NODE_COLS if c in sub.columns]].dropna()
    if len(sub_clean) < 30:
        continue
    notears_csi = div_results.get(dc, {}).get('csi_to_anc4plus', None)
    g_res = run_golem_division(sub_clean, dn)
    l_res = run_lingam_division(sub_clean, dn)
    if dc > 0:
        golem_results[dc] = g_res
        lingam_results[dc] = l_res
    row = {'Division': dn, 'NOTEARS_CSI_ANC': round(notears_csi, 4) if notears_csi is not None else 'NA', 'NOTEARS_pathway': notears_csi is not None and notears_csi > 0.01, 'GOLEM_CSI_ANC': round(g_res['csi_to_anc4plus'], 4) if g_res else 'NA', 'GOLEM_pathway': g_res is not None and g_res['csi_to_anc4plus'] > 0.01, 'LiNGAM_CSI_ANC': round(l_res['csi_to_anc4plus'], 4) if l_res else 'NA', 'LiNGAM_pathway': l_res is not None and l_res['csi_to_anc4plus'] > 0.01}
    votes = [int(row['NOTEARS_pathway']), int(row['GOLEM_pathway']), int(row['LiNGAM_pathway'])]
    votes_valid = [v for v, r in zip(votes, [notears_csi, g_res, l_res]) if r is not None and r != 'NA']
    row['Consensus_votes'] = sum(votes_valid)
    row['N_algorithms'] = len(votes_valid)
    row['Consensus_pathway'] = sum(votes_valid) >= 2
    consensus_rows.append(row)
    print(f'  {dn}: NOTEARS={row['NOTEARS_pathway']}  GOLEM={row['GOLEM_pathway']}  LiNGAM={row['LiNGAM_pathway']}  consensus={row['Consensus_pathway']} ({row['Consensus_votes']}/{row['N_algorithms']})')
consensus_df = pd.DataFrame(consensus_rows)
print(consensus_df.to_string(index=False))
consensus_df.to_csv(OUTPUT_DIR / 'table_cross_algorithm_consensus.csv', index=False)
with open(OUTPUT_DIR / 'cross_algorithm_consensus.json', 'w') as f:
    json.dump(consensus_rows, f, indent=2, default=str)


## Section 8 — Layer 4a: MissForest Imputation


In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

def missforest_impute(df, max_iter=3, label=''):
    df_imp = df.copy()
    for col in df_imp.columns:
        if df_imp[col].isna().any():
            df_imp[col] = df_imp[col].fillna(df_imp[col].median())
    for it in range(max_iter):
        for col in df_imp.columns:
            miss = df[col].isna()
            if miss.sum() == 0:
                continue
            feats = [c for c in df_imp.columns if c != col]
            X_tr = df_imp.loc[~miss, feats]
            y_tr = df_imp.loc[~miss, col]
            X_pr = df_imp.loc[miss, feats]
            if len(y_tr) < 10 or len(X_pr) == 0:
                continue
            rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42) if y_tr.nunique() <= 5 else RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)
            try:
                rf.fit(X_tr, y_tr)
                df_imp.loc[miss, col] = rf.predict(X_pr)
            except:
                pass
    return df_imp
print(f'CLUSTER_PANEL_IMPUTED already built by Cell 7: {CLUSTER_PANEL_IMPUTED.shape}')
model_cols = [c for c in CAUSAL_FEATURES if c in ANALYSIS_2022.columns and c != 'DIVISION']
MODEL_DF_RAW = ANALYSIS_2022[model_cols + ['DIVISION', 'V001']].copy()
print(f'\nPre-imputation missingness (2022):')
print(MODEL_DF_RAW[model_cols].isna().mean().mul(100).round(1).to_string())
MODEL_DF = missforest_impute(MODEL_DF_RAW[model_cols].copy(), max_iter=3, label='2022')
MODEL_DF['DIVISION'] = MODEL_DF_RAW['DIVISION'].values
MODEL_DF['CLUSTER_ID'] = MODEL_DF_RAW['V001'].values
print(f'Post-imputation nulls: {MODEL_DF[model_cols].isna().sum().sum()}')
MODEL_DF.to_parquet(OUTPUT_DIR / 'model_df_2022_imputed.parquet')


## Section 9 — Layer 4b: VAE Vulnerability Profiling


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
device = torch.device('cpu')
torch.set_default_dtype(torch.float32)
BINARY_COLS = [c for c in ['URBAN', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'DIST_HEALTH'] if c in MODEL_DF.columns]
CONTINUOUS_COLS = [c for c in CAUSAL_NODE_COLS if c in MODEL_DF.columns and c not in BINARY_COLS and (c not in ['STUNTING', 'ANC_4PLUS', 'INST_DELIV'])]
vae_features = CONTINUOUS_COLS + BINARY_COLS
N_CONTINUOUS = len(CONTINUOUS_COLS)
N_BINARY = len(BINARY_COLS)
print(f'Continuous ({N_CONTINUOUS}): {CONTINUOUS_COLS}')
print(f'Binary     ({N_BINARY}):     {BINARY_COLS}')
scaler_vae = StandardScaler()
X_cont = scaler_vae.fit_transform(MODEL_DF[CONTINUOUS_COLS].values.astype(np.float32))
X_bin = MODEL_DF[BINARY_COLS].values.astype(np.float32)
X_vae_scaled = np.hstack([X_cont, X_bin]).astype(np.float32)
LATENT_DIM = 16
INPUT_DIM = X_vae_scaled.shape[1]
print(f'Input dim={INPUT_DIM}  Latent dim={LATENT_DIM}')

class TabularVAE(nn.Module):

    def __init__(self, in_dim, lat_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Linear(128, 64), nn.ReLU())
        self.mu_layer = nn.Linear(64, lat_dim)
        self.logvar_layer = nn.Linear(64, lat_dim)
        self.decoder = nn.Sequential(nn.Linear(lat_dim, 64), nn.ReLU(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, in_dim))

    def encode(self, x):
        h = self.encoder(x)
        return (self.mu_layer(h), self.logvar_layer(h))

    def reparameterise(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return (self.decoder(self.reparameterise(mu, logvar)), mu, logvar)

def vae_loss_mixed(recon, x, mu, logvar, n_cont, n_bin):
    mse = nn.functional.mse_loss(recon[:, :n_cont], x[:, :n_cont], reduction='sum')
    bce = nn.functional.binary_cross_entropy_with_logits(recon[:, n_cont:], x[:, n_cont:], reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (mse + bce + kl) / x.size(0)
X_tensor = torch.tensor(X_vae_scaled, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_tensor), batch_size=512, shuffle=True, drop_last=True)
vae = TabularVAE(INPUT_DIM, LATENT_DIM)
opt = torch.optim.Adam(vae.parameters(), lr=0.001)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
N_EPOCHS = 60
losses = []
for epoch in range(N_EPOCHS):
    vae.train()
    ep_loss = 0
    for batch, in loader:
        opt.zero_grad()
        recon, mu, logvar = vae(batch)
        loss = vae_loss_mixed(recon, batch, mu, logvar, N_CONTINUOUS, N_BINARY)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        opt.step()
        ep_loss += loss.item() * batch.size(0)
    sched.step()
    ep_loss /= len(X_tensor)
    losses.append(ep_loss)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch + 1:>3}/{N_EPOCHS}  Loss={ep_loss:.4f}')
vae.eval()
with torch.no_grad():
    mu_vecs, _ = vae.encode(X_tensor)
    Z = mu_vecs.numpy()
print(f'Latent: {Z.shape}  std range [{Z.std(axis=0).min():.3f}, {Z.std(axis=0).max():.3f}]')
silhouettes = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(Z)
    sil = silhouette_score(Z, lbl, sample_size=min(5000, len(Z)))
    silhouettes[k] = sil
    print(f'  k={k}: silhouette={sil:.4f}')
sil_range = max(silhouettes.values()) - min(silhouettes.values())
best_k = max(silhouettes, key=silhouettes.get)
FIXED_K = 3
if best_k != FIXED_K:
    print(f'Silhouette optimum k={best_k}; fixing to k={FIXED_K} for reproducibility')
    best_k = FIXED_K
print(f'\nOptimal k = {best_k}  (silhouette = {silhouettes.get(best_k, 'override'):.4f})')
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
MODEL_DF['VAE_CLUSTER'] = km_final.fit_predict(Z)
profile_cols = vae_features + [c for c in ['STUNTING', 'ANC_4PLUS', 'INST_DELIV'] if c in MODEL_DF.columns]
cluster_profiles = MODEL_DF.groupby('VAE_CLUSTER')[profile_cols].mean().round(3)
print(cluster_profiles.to_string())
print('\nDIST_HEALTH std per cluster (should be >0):')
print(MODEL_DF.groupby('VAE_CLUSTER')['DIST_HEALTH'].std().round(3))
cluster_profiles.to_csv(OUTPUT_DIR / 'vae_cluster_profiles.csv')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(losses, color='#2196F3', linewidth=2)
axes[0].set(xlabel='Epoch', ylabel='ELBO Loss (MSE+BCE+KL)', title='VAE Training Curve')
axes[1].bar(list(silhouettes.keys()), list(silhouettes.values()), color=['#4CAF50' if k == best_k else '#90CAF9' for k in silhouettes.keys()])
axes[1].set(xlabel='k', ylabel='Silhouette', title=f'k=2..7 → optimal k={best_k}')
if 'STUNTING' in MODEL_DF.columns:
    cs = MODEL_DF.groupby('VAE_CLUSTER')['STUNTING'].mean() * 100
    cz = MODEL_DF['VAE_CLUSTER'].value_counts().sort_index()
    ax2 = axes[2].twinx()
    axes[2].bar(cs.index, cs.values, color='#FF5722', alpha=0.7)
    ax2.bar(cz.index, cz.values, color='#2196F3', alpha=0.3)
    axes[2].set(xlabel='VAE Cluster', ylabel='Stunting %', title='Stunting by Profile')
    ax2.set_ylabel('Cluster Size')
plt.suptitle(f'Layer 4b: VAE Profiling (k={best_k}, mixed loss, LATENT=16, 60 epochs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig4_vae_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 10 — Layer 5: Synthetic Population Generation


In [ ]:
import os as _os
import torch
from ctgan import CTGAN
from scipy.stats import wasserstein_distance
_os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.cuda.is_available = lambda: False
torch.set_default_dtype(torch.float32)
gan_features = vae_features
DISCRETE_COLS = [c for c in ['URBAN', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'DIST_HEALTH'] if c in gan_features]
GAN_DF = MODEL_DF[gan_features].astype(np.float32).copy()
print(f'Discrete: {DISCRETE_COLS}')
ctgan = CTGAN(epochs=300, batch_size=500, generator_dim=(256, 256), discriminator_dim=(256, 256), generator_lr=0.0002, discriminator_lr=0.0002, discriminator_steps=1, pac=10, verbose=True)
ctgan.fit(GAN_DF, discrete_columns=DISCRETE_COLS)
N_SYNTHETIC = 180000
SYNTH_DF = ctgan.sample(N_SYNTHETIC)
w_dist = {}
for col in gan_features:
    r = GAN_DF[col].dropna().values
    s = SYNTH_DF[col].dropna().values
    if len(r) > 1000 and len(s) > 1000:
        w_dist[col] = wasserstein_distance(np.random.choice(r, 5000), np.random.choice(s, 5000))
mean_w = np.mean(list(w_dist.values())) if w_dist else 0.0
rc = GAN_DF.corr().fillna(0)
sc = SYNTH_DF[[c for c in gan_features if c in SYNTH_DF]].corr().fillna(0)
cm = [c for c in gan_features if c in sc.columns]
pcmad = np.mean(np.abs(rc.loc[cm, cm].values - sc.loc[cm, cm].values)) if len(cm) > 1 else 0.0
fidelity_tbl = pd.DataFrame({'Feature': gan_features, 'Real Mean': [GAN_DF[c].mean() for c in gan_features], 'Synth Mean': [SYNTH_DF[c].mean() if c in SYNTH_DF else np.nan for c in gan_features], 'Wasserstein': [w_dist.get(c, np.nan) for c in gan_features]}).round(4)
print(f'\nMean W1={mean_w:.4f}  PCMAD={pcmad:.4f}')
print(fidelity_tbl.to_string(index=False))
poor = fidelity_tbl[fidelity_tbl['Wasserstein'] > 0.4]
if not poor.empty:
    print(f'\n️  W1>0.40 (unsuitable for absolute estimation):\n{poor.to_string()}')
fidelity_tbl.to_csv(OUTPUT_DIR / 'synthetic_fidelity.csv', index=False)
SYNTH_DF.to_parquet(OUTPUT_DIR / 'synthetic_population_180k.parquet')
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ['CLIMATE_SHOCK_IDX', 'EDU_YRS', 'WEALTH_Q', 'DIST_HEALTH']):
    if col in GAN_DF and col in SYNTH_DF:
        ax.hist(GAN_DF[col].dropna(), bins=40, alpha=0.6, color='#2196F3', density=True, label='Real')
        ax.hist(SYNTH_DF[col].dropna(), bins=40, alpha=0.6, color='#FF5722', density=True, label='Synth')
        ax.set_title(f'{col}\nW1={w_dist.get(col, 0):.3f}')
        ax.legend(fontsize=7)
plt.suptitle(f'Layer 5: CTGAN Fidelity (Mean W1={mean_w:.3f}, PCMAD={pcmad:.3f})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig5_ctgan.png', dpi=150, bbox_inches='tight')
plt.show()
try:
    from ctgan import TVAE
    import inspect as _inspect
    _tvae_params = set(_inspect.signature(TVAE.__init__).parameters.keys())
    if 'compress_dims' in _tvae_params:
        tvae = TVAE(epochs=300, batch_size=500, compress_dims=(256, 256), decompress_dims=(256, 256), l2scale=1e-05, loss_factor=2, cuda=False)
    elif 'encoder_dim' in _tvae_params:
        tvae = TVAE(epochs=300, batch_size=500, encoder_dim=(256, 256), decoder_dim=(256, 256), l2scale=1e-05, loss_factor=2, cuda=False)
    else:
        tvae = TVAE(epochs=300, batch_size=500, cuda=False)
    tvae.fit(GAN_DF, discrete_columns=DISCRETE_COLS)
    SYNTH_TVAE = tvae.sample(N_SYNTHETIC)
    tvae_w = {}
    for col in gan_features:
        r = GAN_DF[col].dropna().values
        s = SYNTH_TVAE[col].dropna().values
        if len(r) > 1000 and len(s) > 1000:
            from scipy.stats import wasserstein_distance as _wd
            tvae_w[col] = _wd(np.random.choice(r, 5000), np.random.choice(s, 5000))
    mean_w_tvae = np.mean(list(tvae_w.values())) if tvae_w else np.nan
    print(f'  CTGAN mean W1={mean_w:.4f}  TVAE mean W1={mean_w_tvae:.4f}')
    print(f'  {('TVAE' if mean_w_tvae < mean_w else 'CTGAN')} achieves lower mean W1 on this dataset.')
    tvae_comparison = {'ctgan_mean_w1': round(mean_w, 4), 'tvae_mean_w1': round(mean_w_tvae, 4), 'winner': 'TVAE' if mean_w_tvae < mean_w else 'CTGAN', 'per_feature': {col: {'ctgan': round(w_dist.get(col, np.nan), 4), 'tvae': round(tvae_w.get(col, np.nan), 4)} for col in gan_features}}
    with open(OUTPUT_DIR / 'tvae_vs_ctgan.json', 'w') as f:
        json.dump(tvae_comparison, f, indent=2, default=str)
    if mean_w_tvae < mean_w:
        SYNTH_DF_BEST = SYNTH_TVAE
        BEST_SYNTH_MODEL = 'TVAE'
    else:
        SYNTH_DF_BEST = SYNTH_DF
        BEST_SYNTH_MODEL = 'CTGAN'
    print(f'  Selected: {BEST_SYNTH_MODEL}')
except Exception as e:
    tvae_comparison = {'error': str(e), 'ctgan_mean_w1': round(mean_w, 4)}
    SYNTH_DF_BEST = SYNTH_DF
    BEST_SYNTH_MODEL = 'CTGAN'
    with open(OUTPUT_DIR / 'tvae_vs_ctgan.json', 'w') as f:
        json.dump(tvae_comparison, f, indent=2)
try:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    import lightgbm as _lgb
    real_sample = GAN_DF.sample(min(5000, len(GAN_DF)), random_state=42).copy()
    synth_sample = SYNTH_DF[gan_features].sample(min(5000, len(SYNTH_DF)), random_state=42).copy()
    detect_X = pd.concat([real_sample, synth_sample], ignore_index=True).fillna(0)
    detect_y = np.array([1] * len(real_sample) + [0] * len(synth_sample))
    X_tr, X_te, y_tr, y_te = train_test_split(detect_X, detect_y, test_size=0.3, random_state=42, stratify=detect_y)
    clf = _lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    detect_auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    print(f'  Detectability AUC (CTGAN): {detect_auc:.4f}')
    print(f'  (0.5 = perfectly indistinguishable; 1.0 = perfectly detectable)')
    if detect_auc > 0.8:
        pass
    elif detect_auc > 0.65:
        pass
    else:
        pass
    detectability_result = {'ctgan_detect_auc': round(detect_auc, 4), 'interpretation': '0.5=indistinguishable, 1.0=easily_detected'}
    with open(OUTPUT_DIR / 'synthetic_detectability.json', 'w') as f:
        json.dump(detectability_result, f, indent=2)
except Exception as e:
    pass


## Section 11 — Layer 5b: LightGBM + SHAP


In [ ]:
from sklearn.metrics import roc_auc_score, classification_report
if 'HEALTHCARE_EXCLUSION' not in MODEL_DF.columns:
    parts = []
    if 'DIST_HEALTH' in MODEL_DF.columns:
        parts.append(MODEL_DF['DIST_HEALTH'].fillna(0))
    if 'ANC_4PLUS' in MODEL_DF.columns:
        parts.append(1 - MODEL_DF['ANC_4PLUS'].fillna(0))
    if 'INST_DELIV' in MODEL_DF.columns:
        parts.append(1 - MODEL_DF['INST_DELIV'].fillna(0))
    MODEL_DF['HEALTHCARE_EXCLUSION'] = np.mean(parts, axis=0) if parts else 0.5
MODEL_DF['EXCL_BINARY'] = (MODEL_DF['HEALTHCARE_EXCLUSION'] >= MODEL_DF['HEALTHCARE_EXCLUSION'].quantile(0.75)).astype(int)
ml_target = 'EXCL_BINARY'
EXCLUDE = {'STUNTING', 'ANC_4PLUS', 'INST_DELIV', 'DIST_HEALTH', 'DECISION_AUT', 'DECISION_AUTONOMY', 'DIVISION', 'VAE_CLUSTER', 'CLUSTER_ID', 'WAVE', 'HEALTHCARE_EXCLUSION', 'EXCL_BINARY'}
ml_features = [c for c in CAUSAL_NODE_COLS if c in MODEL_DF.columns and c not in EXCLUDE]
print(f'SHAP predictors ({len(ml_features)}): {ml_features}')
ml_X = MODEL_DF[ml_features].fillna(0).astype(np.float32)
ml_y = MODEL_DF[ml_target].fillna(0).astype(int)
from sklearn.model_selection import StratifiedKFold
cluster_ids = MODEL_DF['CLUSTER_ID'].values
unique_c = np.unique(cluster_ids)
clust_label = pd.Series(ml_y.values, index=cluster_ids).groupby(cluster_ids).mean().reindex(unique_c).fillna(0).values
clust_strat = (clust_label > 0.5).astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs, fold_models, fold_shap_vals, fold_X_te_all, fold_y_te_all, fold_te_masks = ([], [], [], [], [], [])
lgbm_params = dict(n_estimators=500, learning_rate=0.05, num_leaves=63, min_child_samples=20, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
for fold, (tr_ci, te_ci) in enumerate(skf.split(unique_c, clust_strat)):
    train_c = set(unique_c[tr_ci])
    test_c = set(unique_c[te_ci])
    tr_mask = np.array([c in train_c for c in cluster_ids])
    te_mask = ~tr_mask
    X_tr_f, X_te_f = (ml_X.iloc[tr_mask], ml_X.iloc[te_mask])
    y_tr_f, y_te_f = (ml_y.iloc[tr_mask], ml_y.iloc[te_mask])
    if 'SURVEY_WEIGHT' in MODEL_DF.columns:
        sw_tr = MODEL_DF['SURVEY_WEIGHT'].iloc[tr_mask].fillna(1.0).values
        sw_te = MODEL_DF['SURVEY_WEIGHT'].iloc[te_mask].fillna(1.0).values
    else:
        sw_tr = np.ones(tr_mask.sum())
        sw_te = np.ones(te_mask.sum())
    tr_he = MODEL_DF.loc[tr_mask, 'HEALTHCARE_EXCLUSION']
    thresh = tr_he.quantile(0.75)
    y_tr_f = (MODEL_DF.loc[tr_mask, 'HEALTHCARE_EXCLUSION'] >= thresh).astype(int)
    y_te_f = (MODEL_DF.loc[te_mask, 'HEALTHCARE_EXCLUSION'] >= thresh).astype(int)
    m = lgb.LGBMClassifier(**lgbm_params)
    m.fit(X_tr_f, y_tr_f, sample_weight=sw_tr, eval_set=[(X_te_f, y_te_f)], eval_sample_weight=[sw_te], callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    prob_f = m.predict_proba(X_te_f)[:, 1]
    auc_f = roc_auc_score(y_te_f, prob_f)
    fold_aucs.append(auc_f)
    fold_models.append(m)
    fold_X_te_all.append(X_te_f)
    fold_y_te_all.append(y_te_f)
    fold_te_masks.append(te_mask)
    print(f'  Fold {fold + 1}/5: {tr_mask.sum():,} train / {te_mask.sum():,} test  AUC={auc_f:.4f}')
auc = float(np.mean(fold_aucs))
auc_std = float(np.std(fold_aucs))
print(f'\nMean AUC-ROC (5-fold cluster-CV): {auc:.4f} ± {auc_std:.4f}')
best_fold = int(np.argmin(np.abs(np.array(fold_aucs) - auc)))
lgbm = fold_models[best_fold]
X_te = fold_X_te_all[best_fold]
y_te = fold_y_te_all[best_fold]
y_prob = lgbm.predict_proba(X_te)[:, 1]
print(classification_report(y_te, (y_prob > 0.5).astype(int), target_names=['Low', 'High']))
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_te)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
if 'SURVEY_WEIGHT' in MODEL_DF.columns:
    _te_mask_best = fold_te_masks[best_fold]
    _sw_shap = MODEL_DF['SURVEY_WEIGHT'].iloc[_te_mask_best].fillna(1.0).values
    _sw_shap = _sw_shap / _sw_shap.sum()
    _mean_shap = np.average(np.abs(shap_values), axis=0, weights=_sw_shap)
else:
    _mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': ml_features, 'Mean |SHAP|': _mean_shap}).sort_values('Mean |SHAP|', ascending=False)
print('\nTop-10 SHAP:')
print(shap_df.head(10).to_string(index=False))
shap_df.to_csv(OUTPUT_DIR / 'shap_feature_importance.csv', index=False)
shap.summary_plot(shap_values, X_te, plot_type='bar', max_display=12, show=False)
plt.title(f'SHAP — Healthcare Exclusion (AUC={auc:.3f}, cluster split)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig6_shap.png', dpi=160, bbox_inches='tight')
plt.show()


## Section 12 — Layer 6: PPO Sequential Policy Optimizer


In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback
import torch
device = 'cpu'
torch.set_default_dtype(torch.float32)
INTERVENTIONS_BASE = [{'name': 'Education Subsidy', 'cost': 0.1, 'edu_delta': 0.15, 'wealth_delta': 0.05}, {'name': 'Mobile Connectivity', 'cost': 0.08, 'mobile_delta': 0.25, 'internet_delta': 0.1}, {'name': 'Sanitation Upgrade', 'cost': 0.12, 'toilet_delta': 0.2, 'water_delta': 0.1}, {'name': 'Healthcare Facility', 'cost': 0.15, 'dist_delta': -0.2, 'anc_delta': 0.15}, {'name': 'Community Health Worker', 'cost': 0.07, 'anc_delta': 0.12, 'inst_delta': 0.1}, {'name': 'Digital Literacy', 'cost': 0.06, 'internet_delta': 0.15, 'mobile_delta': 0.1}, {'name': 'Conditional Cash Transfer', 'cost': 0.14, 'wealth_delta': 0.12, 'edu_delta': 0.08}, {'name': 'Climate Adaptation Infra', 'cost': 0.13, 'climate_delta': -0.2, 'anc_delta': 0.05}]
N_ACTIONS = len(INTERVENTIONS_BASE)
PATHWAY_DIVISIONS = set()
if 'div_results' in dir():
    for _dc, _res in div_results.items():
        if _dc > 0 and _res.get('csi_to_anc4plus', 0) > 0.01:
            PATHWAY_DIVISIONS.add(_dc)
if not PATHWAY_DIVISIONS:
    PATHWAY_DIVISIONS = {5}
print(f'Pathway-encoded divisions (CAI anc_delta=0.18): {[DIVISIONS.get(dc, dc) for dc in sorted(PATHWAY_DIVISIONS)]}')
RAJSHAHI_CODE = 5
INDICATOR_WEIGHTS = {'ANC_4PLUS': 0.2, 'INST_DELIV': 0.15, 'STUNTING': -0.2, 'DIST_HEALTH': -0.1, 'EDU_YRS': 0.08, 'WEALTH_Q': 0.07, 'MOBILE_OWN': 0.05, 'IMPROVED_TOILET': 0.05, 'INTERNET': 0.04, 'IMPROVED_WATER': 0.03, 'CLIMATE_SHOCK_IDX': -0.1}
STATE_COLS_CANDIDATES = ['EDU_YRS', 'WEALTH_Q', 'MOBILE_OWN', 'INTERNET', 'IMPROVED_WATER', 'IMPROVED_TOILET', 'DIST_HEALTH', 'ANC_4PLUS', 'INST_DELIV', 'CLIMATE_SHOCK_IDX', 'STUNTING']

def compute_health_score(state, cols):
    return sum((abs(w) * (1 - np.clip(float(state[cols.index(col)]), 0, 1)) if w < 0 else w * np.clip(float(state[cols.index(col)]), 0, 1) for col, w in INDICATOR_WEIGHTS.items() if col in cols))

class DHSPolicyEnv(gym.Env):

    def __init__(self, df, division_code=0, budget=1.0, max_steps=8):
        super().__init__()
        self.df = df.reset_index(drop=True).copy()
        self.division_code = division_code
        self.budget_max = budget
        self.max_steps = max_steps
        self.interventions = []
        for iv in INTERVENTIONS_BASE:
            d = dict(iv)
            if d['name'] == 'Climate Adaptation Infra' and division_code in PATHWAY_DIVISIONS:
                d['anc_delta'] = 0.18
            self.interventions.append(d)
        self.state_cols = [c for c in STATE_COLS_CANDIDATES if c in self.df.columns]
        self.observation_space = spaces.Box(low=-5, high=5, shape=(len(self.state_cols) + 1,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.scaler = StandardScaler()
        self.df_scaled = pd.DataFrame(self.scaler.fit_transform(self.df[self.state_cols].fillna(0)), columns=self.state_cols)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.budget_left = self.budget_max
        self.actions_used = []
        idx = np.random.randint(0, len(self.df_scaled))
        self.state = self.df_scaled.iloc[idx].values.astype(np.float32)
        self.real_state = self.df[self.state_cols].iloc[idx].values.astype(np.float64).copy()
        self.prev_score = compute_health_score(self.real_state, self.state_cols)
        return (np.append(self.state, self.budget_left / self.budget_max).astype(np.float32), {})

    def step(self, action):
        iv = self.interventions[action]
        cost = iv['cost']
        nrep = self.actions_used.count(action)
        rpen = -0.15 * nrep if nrep > 0 else 0.0
        if cost > self.budget_left + 1e-06:
            reward = -0.5 + rpen
            self.budget_left = max(0.0, self.budget_left - 0.01)
        else:
            self.budget_left -= cost
            col_map = {'edu_delta': 'EDU_YRS', 'wealth_delta': 'WEALTH_Q', 'mobile_delta': 'MOBILE_OWN', 'internet_delta': 'INTERNET', 'toilet_delta': 'IMPROVED_TOILET', 'water_delta': 'IMPROVED_WATER', 'dist_delta': 'DIST_HEALTH', 'anc_delta': 'ANC_4PLUS', 'inst_delta': 'INST_DELIV', 'climate_delta': 'CLIMATE_SHOCK_IDX'}
            for dk, cn in col_map.items():
                if dk in iv and cn in self.state_cols:
                    i = self.state_cols.index(cn)
                    self.real_state[i] = np.clip(self.real_state[i] + iv[dk], 0, 1)
            self.state = np.clip(self.scaler.transform(self.real_state.reshape(1, -1)).flatten().astype(np.float32), -5, 5)
            new_score = compute_health_score(self.real_state, self.state_cols)
            improvement = new_score - self.prev_score
            div_bonus = 0.05 if action not in self.actions_used else 0.0
            reward = improvement * 5 + improvement / (cost + 1e-06) * 0.5 + div_bonus + rpen
            self.prev_score = new_score
            self.actions_used.append(action)
        self.step_count += 1
        done = self.step_count >= self.max_steps or self.budget_left < 0.05
        return (np.append(self.state, self.budget_left / self.budget_max).astype(np.float32), reward, done, False, {})

class RewardCallback(BaseCallback):

    def __init__(self, total, div_name, every=50000):
        super().__init__()
        self.total = total
        self.div_name = div_name
        self.every = every
        self.last = 0
        self.rewards = []

    def _on_step(self):
        if 'rewards' in self.locals:
            self.rewards.extend(self.locals['rewards'].tolist())
        if self.num_timesteps - self.last >= self.every:
            print(f'    {self.div_name}: {self.num_timesteps:>7,}/{self.total:,}  avg_r={np.mean(self.rewards[-200:]):.4f}')
            self.last = self.num_timesteps
        return True
N_ENVS = 8
TIMESTEPS = 300000
PPO_KWARGS = dict(device=device, verbose=0, n_steps=512, batch_size=512, n_epochs=10, learning_rate=0.0003, ent_coef=0.01, clip_range=0.2, policy_kwargs={'net_arch': [256, 256]})
_pw_names = [DIVISIONS.get(dc, dc) for dc in sorted(PATHWAY_DIVISIONS)]
print(f'Pathway-encoded divisions (CAI anc_delta=0.18): {_pw_names}\n')
rl_results = {}
for div_code, div_name in DIVISIONS.items():
    div_df = MODEL_DF[MODEL_DF['DIVISION'] == div_code].copy()
    if len(div_df) < 30:
        div_df = MODEL_DF.copy()
    vec_env = DummyVecEnv([lambda d=div_df.copy(), dc=div_code: DHSPolicyEnv(d, division_code=dc) for _ in range(N_ENVS)])
    vec_env = VecMonitor(vec_env)
    cb = RewardCallback(TIMESTEPS, div_name)
    model = PPO('MlpPolicy', vec_env, seed=42, **PPO_KWARGS)
    model.learn(total_timesteps=TIMESTEPS, callback=cb, progress_bar=False)
    vec_env.close()
    env = DHSPolicyEnv(div_df, division_code=div_code)
    obs, _ = env.reset()
    seq = []
    used = set()
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            feat = model.policy.features_extractor(obs_t)
            lat = model.policy.mlp_extractor(feat)[0]
            logits = model.policy.action_net(lat).cpu().numpy().flatten()
        for u in used:
            logits[u] = -np.inf
        if np.all(logits == -np.inf):
            break
        a = int(np.argmax(logits))
        used.add(a)
        seq.append(env.interventions[a]['name'])
        obs, _, done, _, _ = env.step(a)
    print(f'  Sequence: {' → '.join(seq[:4])}')
    rl_results[div_code] = {'division': div_name, 'optimal_sequence': seq, 'unique_actions': len(set(range(len(seq)))), 'training_rewards': cb.rewards[-500:], 'model': model}

def evaluate_policy(policy_str_or_model, eval_df, div_code, n_eval=200):
    rewards = []
    for _ in range(n_eval):
        env = DHSPolicyEnv(eval_df, division_code=div_code)
        obs, _ = env.reset()
        total = 0
        step = 0
        done = False
        while not done:
            if policy_str_or_model == 'random':
                action = env.action_space.sample()
            elif policy_str_or_model == 'national_uniform':
                action = 3 if step == 0 else step % N_ACTIONS
            else:
                action = int(policy_str_or_model.predict(obs, deterministic=True)[0])
            obs, r, done, _, _ = env.step(action)
            total += r
            step += 1
        rewards.append(total)
    return (float(np.mean(rewards)), float(np.std(rewards)))
eval_df = MODEL_DF.copy()
rand_rs, unif_rs, rl_rs = ([], [], [])
for dc, rr in rl_results.items():
    div_eval_df = eval_df[eval_df['DIVISION'] == dc] if 'DIVISION' in eval_df.columns else eval_df
    if len(div_eval_df) < 10:
        div_eval_df = eval_df
    r_r, _ = evaluate_policy('random', div_eval_df, dc, n_eval=25)
    r_u, _ = evaluate_policy('national_uniform', div_eval_df, dc, n_eval=25)
    r_rl, _ = evaluate_policy(rr['model'], div_eval_df, dc, n_eval=25)
    rand_rs.append(r_r)
    unif_rs.append(r_u)
    rl_rs.append(r_rl)
random_reward = float(np.mean(rand_rs))
random_sd = float(np.std(rand_rs))
uniform_reward = float(np.mean(unif_rs))
uniform_sd = float(np.std(unif_rs))
rl_reward = float(np.mean(rl_rs))
rl_sd = float(np.std(rl_rs))
print(f'  Random:          {random_reward:.4f} ± {random_sd:.3f}')
print(f'  National-uniform:{uniform_reward:.4f} ± {uniform_sd:.3f}  (HCF first)')
print(f'  RL-PPO:          {rl_reward:.4f} ± {rl_sd:.3f}')
print(f'  vs Random:   {(rl_reward - random_reward) / abs(random_reward) * 100:+.1f}%')
print(f'  vs Uniform:  {(rl_reward - uniform_reward) / abs(uniform_reward) * 100:+.1f}%')
seq_df = pd.DataFrame([{'Division': r['division'], 'Step 1': r['optimal_sequence'][0] if r['optimal_sequence'] else '—', 'Step 2': r['optimal_sequence'][1] if len(r['optimal_sequence']) > 1 else '—', 'Step 3': r['optimal_sequence'][2] if len(r['optimal_sequence']) > 2 else '—', 'Step 4': r['optimal_sequence'][3] if len(r['optimal_sequence']) > 3 else '—', 'Unique actions': r['unique_actions']} for r in rl_results.values()])
print(seq_df.to_string(index=False))
seq_df.to_csv(OUTPUT_DIR / 'table3_rl_sequences.csv', index=False)

def greedy_cost_effectiveness_rollout(div_df, div_code, n_eval=200):
    """Sort available interventions by expected Δhealth/cost each step."""
    from copy import deepcopy
    rewards = []
    for _ in range(n_eval):
        env = DHSPolicyEnv(div_df, division_code=div_code)
        obs, _ = env.reset()
        total = 0
        done = False
        used = set()
        while not done:
            best_a, best_score = (None, -np.inf)
            for a_idx, iv in enumerate(env.interventions):
                if a_idx in used or iv['cost'] > env.budget_left:
                    continue
                test_state = env.real_state.copy()
                col_map = {'edu_delta': 'EDU_YRS', 'wealth_delta': 'WEALTH_Q', 'mobile_delta': 'MOBILE_OWN', 'internet_delta': 'INTERNET', 'toilet_delta': 'IMPROVED_TOILET', 'water_delta': 'IMPROVED_WATER', 'dist_delta': 'DIST_HEALTH', 'anc_delta': 'ANC_4PLUS', 'inst_delta': 'INST_DELIV', 'climate_delta': 'CLIMATE_SHOCK_IDX'}
                for dk, cn in col_map.items():
                    if dk in iv and cn in env.state_cols:
                        i = env.state_cols.index(cn)
                        test_state[i] = np.clip(test_state[i] + iv[dk], 0, 1)
                delta_h = compute_health_score(test_state, env.state_cols) - env.prev_score
                score = delta_h / (iv['cost'] + 1e-06)
                if score > best_score:
                    best_score = score
                    best_a = a_idx
            if best_a is None:
                break
            obs, r, done, _, _ = env.step(best_a)
            total += r
            used.add(best_a)
        rewards.append(total)
    return (float(np.mean(rewards)), float(np.std(rewards)))
greedy_rs = []
for dc, rr in rl_results.items():
    div_eval_df = eval_df[eval_df['DIVISION'] == dc] if 'DIVISION' in eval_df.columns else eval_df
    if len(div_eval_df) < 10:
        div_eval_df = eval_df
    r_g, _ = greedy_cost_effectiveness_rollout(div_eval_df, dc, n_eval=25)
    greedy_rs.append(r_g)
greedy_reward = float(np.mean(greedy_rs))
greedy_sd = float(np.std(greedy_rs))
print(f'  Random:          {random_reward:.4f} ± {random_sd:.3f}')
print(f'  National-uniform:{uniform_reward:.4f} ± {uniform_sd:.3f}')
print(f'  Greedy C-E:      {greedy_reward:.4f} ± {greedy_sd:.3f}  ← stronger baseline')
print(f'  RL-PPO:          {rl_reward:.4f} ± {rl_sd:.3f}')
print(f'  PPO vs Greedy:   {(rl_reward - greedy_reward) / abs(greedy_reward + 1e-09) * 100:+.1f}%')
print(f'  PPO vs Random:   {(rl_reward - random_reward) / abs(random_reward + 1e-09) * 100:+.1f}%')
baseline_results = {'random': {'mean': round(random_reward, 4), 'sd': round(random_sd, 4)}, 'national_uniform': {'mean': round(uniform_reward, 4), 'sd': round(uniform_sd, 4)}, 'greedy_CE': {'mean': round(greedy_reward, 4), 'sd': round(greedy_sd, 4)}, 'ppo': {'mean': round(rl_reward, 4), 'sd': round(rl_sd, 4)}, 'ppo_vs_random_pct': round((rl_reward - random_reward) / abs(random_reward + 1e-09) * 100, 2), 'ppo_vs_greedy_pct': round((rl_reward - greedy_reward) / abs(greedy_reward + 1e-09) * 100, 2)}
with open(OUTPUT_DIR / 'baseline_comparison.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('\nGreedy cost-effectiveness intervention sequences:')
greedy_sequences = {}
for dc, dn in DIVISIONS.items():
    div_eval_df = eval_df[eval_df['DIVISION'] == dc] if 'DIVISION' in eval_df.columns else eval_df
    if len(div_eval_df) < 10:
        div_eval_df = eval_df
    env = DHSPolicyEnv(div_eval_df, division_code=dc)
    obs, _ = env.reset()
    used = set()
    seq = []
    done = False
    while not done:
        best_a, best_score = (None, -np.inf)
        for a_idx, iv in enumerate(env.interventions):
            if a_idx in used or iv['cost'] > env.budget_left:
                continue
            test_state = env.real_state.copy()
            col_map = {'edu_delta': 'EDU_YRS', 'wealth_delta': 'WEALTH_Q', 'mobile_delta': 'MOBILE_OWN', 'internet_delta': 'INTERNET', 'toilet_delta': 'IMPROVED_TOILET', 'water_delta': 'IMPROVED_WATER', 'dist_delta': 'DIST_HEALTH', 'anc_delta': 'ANC_4PLUS', 'inst_delta': 'INST_DELIV', 'climate_delta': 'CLIMATE_SHOCK_IDX'}
            for dk, cn in col_map.items():
                if dk in iv and cn in env.state_cols:
                    i = env.state_cols.index(cn)
                    test_state[i] = np.clip(test_state[i] + iv[dk], 0, 1)
            delta_h = compute_health_score(test_state, env.state_cols) - env.prev_score
            score = delta_h / (iv['cost'] + 1e-06)
            if score > best_score:
                best_score = score
                best_a = a_idx
        if best_a is None:
            break
        obs, _, done, _, _ = env.step(best_a)
        used.add(best_a)
        seq.append(env.interventions[best_a]['name'])
    greedy_sequences[dn] = seq
    print(f'  {dn}: {' → '.join(seq[:4])}')
pd.DataFrame([{'Division': dn, 'Greedy_Step1': s[0] if s else '', 'Greedy_Step2': s[1] if len(s) > 1 else '', 'Greedy_Step3': s[2] if len(s) > 2 else '', 'Greedy_Step4': s[3] if len(s) > 3 else ''} for dn, s in greedy_sequences.items()]).to_csv(OUTPUT_DIR / 'table_greedy_sequences.csv', index=False)
N_SEEDS = 5
multirun_rewards = {dc: [] for dc in rl_results}
for seed_i in range(N_SEEDS):
    seed_val = seed_i * 17 + 3
    for div_code, div_name in DIVISIONS.items():
        if div_code not in rl_results:
            continue
        div_df = MODEL_DF[MODEL_DF['DIVISION'] == div_code].copy()
        if len(div_df) < 30:
            div_df = MODEL_DF.copy()
        try:
            vec_env = DummyVecEnv([lambda d=div_df.copy(), dc=div_code: DHSPolicyEnv(d, division_code=dc) for _ in range(N_ENVS)])
            vec_env = VecMonitor(vec_env)
            m = PPO('MlpPolicy', vec_env, seed=seed_val, **PPO_KWARGS)
            m.learn(total_timesteps=TIMESTEPS, progress_bar=False)
            vec_env.close()
            r_rl, _ = evaluate_policy(m, div_df, div_code, n_eval=20)
            multirun_rewards[div_code].append(r_rl)
        except Exception as e:
            pass
multirun_summary = []
for dc, dn in DIVISIONS.items():
    rs = multirun_rewards.get(dc, [])
    if rs:
        multirun_summary.append({'Division': dn, 'Mean_reward': round(np.mean(rs), 4), 'SD_reward': round(np.std(rs), 4), 'CI95_lo': round(np.mean(rs) - 1.96 * np.std(rs) / np.sqrt(len(rs)), 4), 'CI95_hi': round(np.mean(rs) + 1.96 * np.std(rs) / np.sqrt(len(rs)), 4), 'N_seeds': len(rs)})
multirun_df = pd.DataFrame(multirun_summary)
print('\nPPO reward uncertainty across 5 random seeds:')
print(multirun_df.to_string(index=False))
multirun_df.to_csv(OUTPUT_DIR / 'table_ppo_multirun_uncertainty.csv', index=False)
with open(OUTPUT_DIR / 'ppo_multirun_uncertainty.json', 'w') as f:
    json.dump(multirun_summary, f, indent=2)
STATE_COLS_NO_CSI = [c for c in STATE_COLS_CANDIDATES if c != 'CLIMATE_SHOCK_IDX']

class DHSPolicyEnvNoCSI(DHSPolicyEnv):
    """PPO environment with CLIMATE_SHOCK_IDX excluded from state observations."""

    def __init__(self, df, division_code=0, budget=1.0, max_steps=8):
        super().__init__(df, division_code, budget, max_steps)
        self.state_cols = [c for c in STATE_COLS_NO_CSI if c in self.df.columns]
        self.observation_space = spaces.Box(low=-5, high=5, shape=(len(self.state_cols) + 1,), dtype=np.float32)
        self.scaler = StandardScaler()
        self.df_scaled = pd.DataFrame(self.scaler.fit_transform(self.df[self.state_cols].fillna(0)), columns=self.state_cols)
csi_excluded_results = {}
for div_code in PATHWAY_DIVISIONS:
    div_name = DIVISIONS.get(div_code, f'Div{div_code}')
    div_df = MODEL_DF[MODEL_DF['DIVISION'] == div_code].copy()
    if len(div_df) < 30:
        div_df = MODEL_DF.copy()
    r_with, _ = evaluate_policy(rl_results[div_code]['model'], div_df, div_code, n_eval=50)
    try:
        vec_env_no_csi = DummyVecEnv([lambda d=div_df.copy(), dc=div_code: DHSPolicyEnvNoCSI(d, division_code=dc) for _ in range(N_ENVS)])
        vec_env_no_csi = VecMonitor(vec_env_no_csi)
        model_no_csi = PPO('MlpPolicy', vec_env_no_csi, seed=42, **PPO_KWARGS)
        model_no_csi.learn(total_timesteps=TIMESTEPS, progress_bar=False)
        vec_env_no_csi.close()
        rewards_no_csi = []
        for _ in range(50):
            env_nc = DHSPolicyEnvNoCSI(div_df, division_code=div_code)
            obs, _ = env_nc.reset()
            total = 0
            done = False
            while not done:
                action = int(model_no_csi.predict(obs, deterministic=True)[0])
                obs, r, done, _, _ = env_nc.step(action)
                total += r
            rewards_no_csi.append(total)
        r_without = float(np.mean(rewards_no_csi))
        env_nc = DHSPolicyEnvNoCSI(div_df, division_code=div_code)
        obs, _ = env_nc.reset()
        seq_no_csi = []
        used = set()
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                feat = model_no_csi.policy.features_extractor(obs_t)
                lat = model_no_csi.policy.mlp_extractor(feat)[0]
                logits = model_no_csi.policy.action_net(lat).cpu().numpy().flatten()
            for u in used:
                logits[u] = -np.inf
            if np.all(logits == -np.inf):
                break
            a = int(np.argmax(logits))
            used.add(a)
            seq_no_csi.append(env_nc.interventions[a]['name'])
            obs, _, done, _, _ = env_nc.step(a)
        cai_step_with = rl_results[div_code]['optimal_sequence'].index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in rl_results[div_code]['optimal_sequence'] else None
        cai_step_without = seq_no_csi.index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in seq_no_csi else None
        csi_excluded_results[div_name] = {'reward_with_csi': round(r_with, 4), 'reward_without_csi': round(r_without, 4), 'seq_with_csi': rl_results[div_code]['optimal_sequence'][:8], 'seq_without_csi': seq_no_csi[:8], 'cai_step_with_csi': cai_step_with, 'cai_step_without_csi': cai_step_without, 'cai_step_changed': cai_step_with != cai_step_without}
        print(f'  {div_name}:')
        print(f'    With CSI:    reward={r_with:.4f}  CAI@step {cai_step_with}  seq={rl_results[div_code]['optimal_sequence'][:4]}')
        print(f'    Without CSI: reward={r_without:.4f}  CAI@step {cai_step_without}  seq={seq_no_csi[:4]}')
        print(f'    CAI step changed: {cai_step_with != cai_step_without}')
    except Exception as e:
        csi_excluded_results[div_name] = {'error': str(e)}
with open(OUTPUT_DIR / 'stronger_ablation_no_csi.json', 'w') as f:
    json.dump(csi_excluded_results, f, indent=2, default=str)
ablation_rows = []
for dn, res in csi_excluded_results.items():
    if 'error' not in res:
        ablation_rows.append({'Division': dn, 'Reward_with_CSI': res.get('reward_with_csi'), 'Reward_without_CSI': res.get('reward_without_csi'), 'CAI_step_with_CSI': res.get('cai_step_with_csi'), 'CAI_step_without_CSI': res.get('cai_step_without_csi'), 'CAI_step_changed': res.get('cai_step_changed')})
pd.DataFrame(ablation_rows).to_csv(OUTPUT_DIR / 'table_stronger_ablation_no_csi.csv', index=False)


## Section 13 — Layer 7: Pareto Optimisation


In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
div_health = {}
for dc, dn in DIVISIONS.items():
    sub_traj = trajectory_df[trajectory_df['DIVISION'] == float(dc)] if 'trajectory_df' in dir() else pd.DataFrame()
    if len(sub_traj) > 0:
        if 'HEALTHCARE_EXCLUSION' not in sub_traj.columns:
            sub_traj = sub_traj.copy()
            sub_traj['HEALTHCARE_EXCLUSION'] = np.mean([sub_traj['DIST_HEALTH'].fillna(0), 1 - sub_traj['ANC_4PLUS'].fillna(0), 1 - sub_traj['INST_DELIV'].fillna(0)], axis=0)
        s = sub_traj['STUNTING'].mean()
        e = sub_traj['HEALTHCARE_EXCLUSION'].mean()
        a = sub_traj['ANC_4PLUS'].mean()
    else:
        sub = MODEL_DF[MODEL_DF['DIVISION'] == dc]
        if 'HEALTHCARE_EXCLUSION' not in sub.columns:
            sub = sub.copy()
            sub['HEALTHCARE_EXCLUSION'] = np.mean([sub['DIST_HEALTH'].fillna(0), 1 - sub['ANC_4PLUS'].fillna(0), 1 - sub['INST_DELIV'].fillna(0)], axis=0)
        s = sub['STUNTING'].mean() if len(sub) > 0 else 0.28
        e = sub['HEALTHCARE_EXCLUSION'].mean() if len(sub) > 0 else 0.35
        a = sub['ANC_4PLUS'].mean() if len(sub) > 0 else 0.3
    div_health[dc] = {'stunting': s, 'exclusion': e, 'anc': a, 'name': dn}
N_DIVS = len(DIVISIONS)

class BDHSHealthPolicy(Problem):

    def __init__(self):
        super().__init__(n_var=N_DIVS, n_obj=2, n_ieq_constr=1, xl=np.full(N_DIVS, 0.02), xu=np.ones(N_DIVS))
        self.div_codes = list(DIVISIONS.keys())
        self.h_base = np.array([div_health[d]['stunting'] * 0.5 + div_health[d]['exclusion'] * 0.5 for d in self.div_codes])

    def _evaluate(self, X, out, *args, **kwargs):
        F = np.zeros((X.shape[0], 2))
        G = np.zeros((X.shape[0], 1))
        for i, x in enumerate(X):
            xn = x / (x.sum() + 1e-09)
            hd = xn * self.h_base
            F[i, 0] = -hd.sum()
            burden = np.clip(self.h_base - hd, 0, 1)
            arr = sorted(burden)
            n = len(arr)
            s = sum(arr)
            F[i, 1] = 2 * sum(((k + 1) * arr[k] for k in range(n))) / (n * s) - (n + 1) / n if s > 0 else 0.0
            G[i, 0] = x.sum() - 1.0
        out['F'] = F
        out['G'] = G
problem = BDHSHealthPolicy()
algorithm = NSGA2(pop_size=100, sampling=FloatRandomSampling(), crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20), eliminate_duplicates=True)
print('NSGA-II: 500 generations, 5 independent runs...')
all_H, all_G = ([], [])
for run in range(5):
    r = minimize(problem, algorithm, get_termination('n_gen', 500), seed=run * 10 + 42, verbose=False)
    pF = r.F
    ideal = pF.min(axis=0)
    nadir = pF.max(axis=0)
    norm = (pF - ideal) / (nadir - ideal + 1e-09)
    ki = int(np.argmin(np.sqrt(norm[:, 0] ** 2 + norm[:, 1] ** 2)))
    all_H.append(-pF[ki, 0])
    all_G.append(pF[ki, 1])
    print(f'  Run {run + 1}: H={-pF[ki, 0]:.4f}  Gini={pF[ki, 1]:.4f}')
print(f'  H range: {min(all_H):.3f}–{max(all_H):.3f}  Gini range: {min(all_G):.3f}–{max(all_G):.3f}')
res = minimize(problem, algorithm, get_termination('n_gen', 500), seed=42, verbose=False)
pareto_F = res.F
pareto_X = res.X
ideal = pareto_F.min(axis=0)
nadir = pareto_F.max(axis=0)
norm = (pareto_F - ideal) / (nadir - ideal + 1e-09)
knee_idx = int(np.argmin(np.sqrt(norm[:, 0] ** 2 + norm[:, 1] ** 2)))
knee_solution = pareto_X[knee_idx]
knee_objectives = pareto_F[knee_idx]
print(f'\nKnee: H={-knee_objectives[0]:.4f}  Gini={knee_objectives[1]:.4f}')
print(dict(zip(DIVISIONS.values(), knee_solution.round(3))))
pd.DataFrame(pareto_F, columns=['neg_H', 'Gini']).assign(Health_Gain=-pareto_F[:, 0]).to_csv(OUTPUT_DIR / 'pareto_frontier.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(-pareto_F[:, 0], pareto_F[:, 1], c=pareto_F[:, 1], cmap='YlOrRd', s=50, alpha=0.8)
axes[0].scatter(-knee_objectives[0], knee_objectives[1], color='#4CAF50', s=200, marker='*', zorder=5, label=f'Knee (H={-knee_objectives[0]:.3f}, Gini={knee_objectives[1]:.3f})')
axes[0].set(xlabel='Health Gain (higher=better)', ylabel='Regional Gini', title='Layer 7: Pareto Frontier (linear objective, 5 runs)')
axes[0].invert_xaxis()
axes[0].legend()
alloc = pd.DataFrame({'Division': list(DIVISIONS.values()), 'Budget': knee_solution.round(3)})
axes[1].barh(alloc['Division'], alloc['Budget'], color=plt.cm.YlOrRd(alloc['Budget'] / alloc['Budget'].max()))
axes[1].set(xlabel='Budget Fraction (Knee)', title='Optimal Budget Allocation')
for i, (_, row) in enumerate(alloc.iterrows()):
    axes[1].text(row['Budget'] + 0.001, i, f'{row['Budget']:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig7_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
SENSITIVITY_PARAMS = {'ANC_delta_HCF': ('Healthcare Facility', 'anc_delta', 0.15), 'ANC_delta_CHW': ('Community Health Worker', 'anc_delta', 0.12), 'Climate_delta_CAI': ('Climate Adaptation Infra', 'climate_delta', -0.2), 'Edu_delta_ES': ('Education Subsidy', 'edu_delta', 0.15)}
PERTURB_FACTORS = [0.5, 1.0, 1.5]
sensitivity_rows = []
import copy as _copy
for param_name, (interv_name, delta_key, base_val) in SENSITIVITY_PARAMS.items():
    for factor in PERTURB_FACTORS:
        perturbed_val = base_val * factor
        patched = []
        for iv in INTERVENTIONS_BASE:
            d = dict(iv)
            if d['name'] == interv_name and delta_key in d:
                d[delta_key] = perturbed_val
            patched.append(d)

        class SensitivityEnv(DHSPolicyEnv):
            PATCHED = patched

            def __init__(self, df, division_code=0, budget=1.0, max_steps=8):
                super().__init__(df, division_code, budget, max_steps)
                self.interventions = [dict(iv) for iv in SensitivityEnv.PATCHED]
                if division_code in PATHWAY_DIVISIONS:
                    for iv in self.interventions:
                        if iv['name'] == 'Climate Adaptation Infra':
                            iv['anc_delta'] = 0.18

        class SensitivityProblem(BDHSHealthPolicy):

            def _evaluate(self, X, out, *args, **kwargs):
                F = np.zeros((X.shape[0], 2))
                G = np.zeros((X.shape[0], 1))
                for i, x in enumerate(X):
                    xn = x / (x.sum() + 1e-09)
                    hd = xn * self.h_base
                    F[i, 0] = -hd.sum()
                    burden = np.clip(self.h_base - hd, 0, 1)
                    arr = sorted(burden)
                    n = len(arr)
                    s = sum(arr)
                    F[i, 1] = 2 * sum(((k + 1) * arr[k] for k in range(n))) / (n * s) - (n + 1) / n if s > 0 else 0.0
                    G[i, 0] = x.sum() - 1.0
                out['F'] = F
                out['G'] = G
        try:
            H_vals, G_vals = ([], [])
            for _run in range(3):
                _r = minimize(SensitivityProblem(), algorithm, get_termination('n_gen', 200), seed=_run * 7 + 42, verbose=False)
                _pF = _r.F
                _ideal = _pF.min(axis=0)
                _nadir = _pF.max(axis=0)
                _norm = (_pF - _ideal) / (_nadir - _ideal + 1e-09)
                _ki = int(np.argmin(np.sqrt(_norm[:, 0] ** 2 + _norm[:, 1] ** 2)))
                H_vals.append(-_pF[_ki, 0])
                G_vals.append(_pF[_ki, 1])
            row = {'Parameter': param_name, 'Intervention': interv_name, 'Delta_key': delta_key, 'Base_val': base_val, 'Perturb_factor': factor, 'Perturbed_val': round(perturbed_val, 4), 'Knee_H_mean': round(np.mean(H_vals), 4), 'Knee_H_sd': round(np.std(H_vals), 4), 'Knee_Gini_mean': round(np.mean(G_vals), 4), 'Knee_Gini_sd': round(np.std(G_vals), 4)}
            sensitivity_rows.append(row)
            print(f'  {param_name} × {factor:.1f}: H={np.mean(H_vals):.4f}±{np.std(H_vals):.4f}  Gini={np.mean(G_vals):.4f}±{np.std(G_vals):.4f}')
        except Exception as e:
            pass
sens_df = pd.DataFrame(sensitivity_rows)
print('\nParameter sensitivity summary:')
print(sens_df.to_string(index=False))
print('  equity-efficiency tradeoff is robust to ±50% uncertainty in heuristic parameters.')
print('  Large sensitivity (large SD or large range across factors) = parameter-dependent conclusions.')
sens_df.to_csv(OUTPUT_DIR / 'table_sensitivity_analysis.csv', index=False)
with open(OUTPUT_DIR / 'sensitivity_analysis.json', 'w') as f:
    json.dump(sensitivity_rows, f, indent=2, default=str)


## Section 14 — Policy Brief Generation


In [ ]:
def safe_pct(v):
    return f"{v:.1%}" if isinstance(v, (int, float)) else "N/A"


def generate_brief(div_name, div_code, top_shap, rl_seq, alloc, climate_str, stats):
    return textwrap.dedent(f"""
    POLICY BRIEF — {div_name.upper()} DIVISION
    Bangladesh Ministry of Health and Family Welfare

    CURRENT BURDEN
    Stunting: {safe_pct(stats.get('stunting', 0.28))} | Exclusion: {safe_pct(stats.get('exclusion', 0.35))} | ANC4+: {safe_pct(stats.get('anc', 0.30))}

    CAUSAL FINDINGS
    {climate_str}
    Top predictors: {', '.join(top_shap['Feature'].head(5).tolist())}

    RECOMMENDED INTERVENTION SEQUENCE
    {chr(10).join(f'  Step {i+1}: {s}' for i, s in enumerate(rl_seq[:4]))}

    BUDGET: {alloc:.1%} of total health budget (Pareto knee-point)

    IMPLEMENTATION
    Months 1-6: Assessment and mobilisation
    Months 7-18: Phase 1 (Steps 1-2)
    Months 19-36: Scale-up and monitoring

    NOTE: Simulation-derived. Requires prospective cluster-randomised validation.
    """).strip()


policy_briefs = {}
div_list = list(DIVISIONS.keys())
for dc, dn in DIVISIONS.items():
    if dc not in div_results:
        continue
    anc_w = div_results[dc]["csi_to_anc4plus"]
    climate_str = (f"CSI→ANC4+ pathway detected (w={anc_w:.3f}); hypothesis-generating, needs validation"
                   if anc_w > 0.01
                   else "No climate pathway detected; socioeconomic factors dominate")
    d_stats = div_health.get(dc, {"stunting": 0.28, "exclusion": 0.35, "anc": 0.30})
    rl_seq = rl_results.get(dc, {}).get("optimal_sequence", ["Healthcare Facility", "CHW"])
    alloc = float(knee_solution[div_list.index(dc)]) if dc in div_list else 0.125
    brief = generate_brief(dn, dc, shap_df, rl_seq, alloc, climate_str, d_stats)
    policy_briefs[dn] = brief
    with open(OUTPUT_DIR / f"policy_brief_{dn.lower()}.txt", "w") as f:
        f.write(brief)

print(f"{len(policy_briefs)} briefs saved.")


## Section 15 — Results Summary & Export


In [ ]:
greedy_reward = greedy_reward if 'greedy_reward' in dir() else float('nan')
pcmad = pcmad if 'pcmad' in dir() else float('nan')
mean_w = mean_w if 'mean_w' in dir() else float('nan')
mean_shd = mean_shd if 'mean_shd' in dir() else float('nan')
auc = auc if 'auc' in dir() else float('nan')
print(' BDHS GENERATIVE CAUSAL AI — FINAL RESULTS SUMMARY')
print('Output files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    sz = f.stat().st_size
    print(f'  {f.name:<45} {('%.0f KB' % (sz / 1024) if sz < 1000000.0 else '%.1f MB' % (sz / 1000000.0))}')
import zipfile as zf
zip_path = '/kaggle/working/bdhs_final.zip'
with zf.ZipFile(zip_path, 'w', zf.ZIP_DEFLATED) as z:
    for fp in OUTPUT_DIR.iterdir():
        z.write(fp, fp.name)


## Section 16 — Publication Figure Panel


In [ ]:
fig = plt.figure(figsize=(22, 20), facecolor='white')
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :2])
if 'STUNTING' in trajectory_df.columns:
    trend = trajectory_df.groupby(['WAVE', 'DIVISION'])['STUNTING'].mean().reset_index()
    for dc, dn in DIVISIONS.items():
        sub = trend[trend['DIVISION'] == dc]
        if len(sub) > 1:
            ax1.plot(sub['WAVE'], sub['STUNTING'] * 100, marker='o', label=dn, linewidth=2)
ax1.set_title('Child Stunting by Division (2011-2022)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Stunting (%)')
ax1.set_xticks([2011, 2014, 2017, 2022])
ax1.legend(fontsize=7, ncol=2)
ax2 = fig.add_subplot(gs[0, 2])
if n_divs > 1:
    sns.heatmap(shd_df, annot=True, fmt='d', cmap='YlOrRd', ax=ax2, linewidths=0.5, cbar=False, annot_kws={'size': 7})
ax2.set_title(f'Pairwise SHD (Mean={mean_shd:.1f})', fontsize=10, fontweight='bold')
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(list(silhouettes.keys()), list(silhouettes.values()), color=['#4CAF50' if k == best_k else '#90CAF9' for k in silhouettes.keys()])
ax3.set_title(f'VAE Silhouette k=2..7 → k={best_k}', fontsize=10, fontweight='bold')
ax3.set_xlabel('k')
ax3.set_ylabel('Silhouette')
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(fidelity_tbl['Feature'][:8], fidelity_tbl['Wasserstein'][:8], color='#FF9800', alpha=0.8)
ax4.set_title(f'CTGAN Fidelity (W1={mean_w:.3f})', fontsize=10, fontweight='bold')
ax4.set_ylabel('Wasserstein-1')
ax4.tick_params(axis='x', rotation=45, labelsize=7)
ax5 = fig.add_subplot(gs[1, 2])
ax5.scatter(-pareto_F[:, 0], pareto_F[:, 1], c=pareto_F[:, 1], cmap='YlOrRd', s=40, alpha=0.8)
ax5.scatter(-knee_objectives[0], knee_objectives[1], color='#4CAF50', s=150, marker='*', zorder=5, label='Knee')
ax5.set_title('Pareto Frontier (linear, 5 runs)', fontsize=10, fontweight='bold')
ax5.set_xlabel('Health Gain')
ax5.set_ylabel('Gini')
ax5.invert_xaxis()
ax5.legend()
ax6 = fig.add_subplot(gs[2, :])
top10 = shap_df.head(10)
ax6.barh(top10['Feature'][::-1], top10['Mean |SHAP|'][::-1], color='#3F51B5', alpha=0.8)
ax6.set_title(f'Top-10 SHAP (AUC={auc:.3f}, cluster-level split, no leakage)', fontsize=11, fontweight='bold')
ax6.set_xlabel('Mean |SHAP|')
ax7 = fig.add_subplot(gs[3, :2])
colors = plt.cm.Set2(np.linspace(0, 1, len(rl_results)))
for (dc, rr), col in zip(rl_results.items(), colors):
    rw = rr['training_rewards']
    if rw:
        ax7.plot(pd.Series(rw).rolling(20, min_periods=1).mean().values, label=rr['division'], color=col, linewidth=1.5)
ax7.set_title('PPO Training Rewards (all 8 divisions, Rajshahi anc_delta=0.18)', fontsize=11, fontweight='bold')
ax7.set_xlabel('Step')
ax7.set_ylabel('Episode reward')
ax7.legend(fontsize=7, ncol=2)
ax8 = fig.add_subplot(gs[3, 2])
ax8.barh(list(DIVISIONS.values()), knee_solution, color=plt.cm.YlOrRd(knee_solution / knee_solution.max() + 0.1))
ax8.set_title('Optimal Budget (Knee)', fontsize=10, fontweight='bold')
ax8.set_xlabel('Budget Fraction')
fig.suptitle('Generative Causal AI — BDHS 2011-2022 (Final Corrected)', fontsize=16, fontweight='bold', y=1.01)
plt.savefig(OUTPUT_DIR / 'fig_full_panel.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


## Section 17 — Additional Robustness Checks and Output Packaging


In [ ]:
import numpy as np, pandas as pd
import logging
import matplotlib.pyplot as plt
SEVERITY_TERCILE = 0.667
B_SUB = 500
_panel = CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel
_pathway_divs = {dc: res for dc, res in div_results.items() if dc > 0 and res.get('csi_to_anc4plus', 0) > 0.01}
if not _pathway_divs:
    pass
else:
    subcluster_results = {}
    for dc, res in _pathway_divs.items():
        dname = DIVISIONS.get(dc, f'Div{dc}')
        sub_full = _panel[_panel['DIVISION'] == float(dc)].copy()
        if 'CLIMATE_SHOCK_IDX' not in sub_full.columns or len(sub_full) < 20:
            continue
        thresh = sub_full['CLIMATE_SHOCK_IDX'].quantile(SEVERITY_TERCILE)
        sub_high = sub_full[sub_full['CLIMATE_SHOCK_IDX'] >= thresh].copy()
        print(f'  {dname}: full n={len(sub_full)}, high-drought subset n={len(sub_high)} (CSI >= {thresh:.2f})')
        if len(sub_high) < 20:
            continue
        boot_ws = []
        np.random.seed(42)
        for _b in range(B_SUB):
            samp = sub_high.sample(n=len(sub_high), replace=True)
            try:
                from castle.algorithms import Notears
                logging.disable(logging.CRITICAL)
                from sklearn.preprocessing import StandardScaler
                xs = StandardScaler().fit_transform(samp[CAUSAL_NODE_COLS].dropna())
                m = Notears(w_threshold=0.01)
                m.learn(xs)
                adj = pd.DataFrame(m.causal_matrix, index=CAUSAL_NODE_COLS, columns=CAUSAL_NODE_COLS)
                logging.disable(logging.NOTSET)
                ci = CAUSAL_NODE_COLS.index('CLIMATE_SHOCK_IDX')
                ai = CAUSAL_NODE_COLS.index('ANC_4PLUS')
                boot_ws.append(float(adj.iloc[ci, ai]))
            except Exception:
                boot_ws.append(0.0)
        arr = np.array(boot_ws)
        pct = float((arr > 0.01).mean() * 100)
        mean_w, sd_w = (float(arr.mean()), float(arr.std()))
        lo, hi = (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
        print(f'    HIGH-DROUGHT SUBSET: mean={mean_w:.3f} SD={sd_w:.3f} 95%CI=[{lo:.3f},{hi:.3f}] positive={pct:.1f}%')
        print(f'    (full-division comparison, from Cell 7: {conv_log.get(dname, {}).get('bootstrap_pct_pos', 'n/a')}% positive)\n')
        subcluster_results[dname] = {'n_full': len(sub_full), 'n_high_drought': len(sub_high), 'csi_threshold': float(thresh), 'bootstrap_mean': round(mean_w, 4), 'bootstrap_sd': round(sd_w, 4), 'bootstrap_95ci_lo': round(lo, 4), 'bootstrap_95ci_hi': round(hi, 4), 'bootstrap_pct_pos': round(pct, 2), 'full_division_pct_pos': conv_log.get(dname, {}).get('bootstrap_pct_pos')}
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.hist(arr, bins=40, color='darkorange', alpha=0.7)
        ax.axvline(mean_w, color='red', lw=2, label=f'Mean={mean_w:.3f}')
        ax.axvline(lo, color='gray', lw=1.2, ls='--', label='2.5/97.5 pct')
        ax.axvline(hi, color='gray', lw=1.2, ls='--')
        ax.set_xlabel('CSI→ANC4+ edge weight')
        ax.set_ylabel('Resamples')
        ax.set_title(f'High-drought subset bootstrap: {dname} (B={B_SUB}, positive={pct:.1f}%)')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'fig_bootstrap_subcluster_{dname.lower()}.png', dpi=150, bbox_inches='tight')
        plt.show()
    import json
    with open(OUTPUT_DIR / 'subcluster_bootstrap_results.json', 'w') as f:
        json.dump(subcluster_results, f, indent=2)
import copy
ABLATION_TIMESTEPS = TIMESTEPS
ablation_results = {}
for div_code in sorted(PATHWAY_DIVISIONS):
    div_name = DIVISIONS.get(div_code, str(div_code))
    print(f'{'─' * 45}\n{div_name} — ABLATION (anc_delta forced to 0.05)\n{'─' * 45}')
    div_df = MODEL_DF[MODEL_DF['DIVISION'] == div_code].copy()
    if len(div_df) < 30:
        div_df = MODEL_DF.copy()

    class DHSPolicyEnvAblation(DHSPolicyEnv):

        def __init__(self, df, division_code=0, budget=1.0, max_steps=8):
            gym.Env.__init__(self)
            self.df = df.reset_index(drop=True).copy()
            self.division_code = division_code
            self.budget_max = budget
            self.max_steps = max_steps
            self.interventions = copy.deepcopy(INTERVENTIONS_BASE)
            self.state_cols = [c for c in STATE_COLS_CANDIDATES if c in self.df.columns]
            self.observation_space = spaces.Box(low=-5, high=5, shape=(len(self.state_cols) + 1,), dtype=np.float32)
            self.action_space = spaces.Discrete(N_ACTIONS)
            self.scaler = StandardScaler()
            self.df_scaled = pd.DataFrame(self.scaler.fit_transform(self.df[self.state_cols].fillna(0)), columns=self.state_cols)
    vec_env = DummyVecEnv([lambda d=div_df.copy(), dc=div_code: DHSPolicyEnvAblation(d, division_code=dc) for _ in range(N_ENVS)])
    vec_env = VecMonitor(vec_env)
    cb = RewardCallback(ABLATION_TIMESTEPS, f'{div_name}-ablation')
    model = PPO('MlpPolicy', vec_env, seed=42, **PPO_KWARGS)
    model.learn(total_timesteps=ABLATION_TIMESTEPS, callback=cb, progress_bar=False)
    vec_env.close()
    env = DHSPolicyEnvAblation(div_df, division_code=div_code)
    obs, _ = env.reset()
    seq = []
    used = set()
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            feat = model.policy.features_extractor(obs_t)
            lat = model.policy.mlp_extractor(feat)[0]
            logits = model.policy.action_net(lat).cpu().numpy().flatten()
        for u in used:
            logits[u] = -np.inf
        if np.all(logits == -np.inf):
            break
        a = int(np.argmax(logits))
        used.add(a)
        seq.append(env.interventions[a]['name'])
        obs, _, done, _, _ = env.step(a)
    original_seq = rl_results.get(div_code, {}).get('optimal_sequence', [])
    cai_step_ablation = seq.index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in seq else None
    cai_step_original = original_seq.index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in original_seq else None
    print(f'  Original (anc_delta=0.18) sequence: {' → '.join(original_seq[:4])}')
    print(f'  Ablation (anc_delta=0.05) sequence:  {' → '.join(seq[:4])}')
    print(f'  CAI step — original: {cai_step_original}, ablation: {cai_step_ablation}\n')
    ablation_results[div_name] = {'original_sequence': original_seq, 'ablation_sequence': seq, 'cai_step_original': cai_step_original, 'cai_step_ablation': cai_step_ablation, 'preference_held_under_ablation': cai_step_ablation == cai_step_original}
import json
with open(OUTPUT_DIR / 'ablation_anc_delta_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
abl_df = pd.DataFrame([{'Division': k, 'CAI step (original, 0.18)': v['cai_step_original'], 'CAI step (ablation, 0.05)': v['cai_step_ablation'], 'Preference held?': v['preference_held_under_ablation']} for k, v in ablation_results.items()])
print(abl_df.to_string(index=False))
abl_df.to_csv(OUTPUT_DIR / 'table_ablation_anc_delta.csv', index=False)
print('   the CAI-first preference is driven by something beyond the reward encoding')
print('   (e.g. climate_delta itself), strengthening the emergent-preference claim.')
print('   If False, the preference is confirmed to be reward-construction-dependent,')
print('   which should be reported plainly in the Discussion rather than as corroboration.')


In [ ]:
import shutil
from pathlib import Path
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = Path('/kaggle/working/bdhs_outputs')
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"{OUTPUT_DIR} does not exist. The kernel was likely fully reset and the files were not persisted. You'll need to re-run Cell 2 (to recreate OUTPUT_DIR) and, if the underlying files are gone, Cells A and B as well.")
ROBUSTNESS_FILES = ['subcluster_bootstrap_results.json', 'ablation_anc_delta_results.json', 'table_ablation_anc_delta.csv']
robustness_figs = list(Path(OUTPUT_DIR).glob('fig_bootstrap_subcluster_*.png'))
context_files = ['notears_convergence_log.json']
context_figs = list(Path(OUTPUT_DIR).glob('fig_bootstrap_*.png'))
context_figs = [f for f in context_figs if 'subcluster' not in f.name]
STAGE_DIR = Path('/kaggle/working/robustness_package') if Path('/kaggle/working').exists() else Path('./robustness_package')
STAGE_DIR.mkdir(parents=True, exist_ok=True)
copied, missing = ([], [])

def _copy(fname_or_path, subfolder=''):
    src = Path(OUTPUT_DIR) / fname_or_path if isinstance(fname_or_path, str) else fname_or_path
    if src.exists():
        dest_dir = STAGE_DIR / subfolder
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest_dir / src.name)
        copied.append(str(src.name))
    else:
        missing.append(str(src))
for f in ROBUSTNESS_FILES:
    _copy(f, subfolder='new_results')
for f in robustness_figs:
    _copy(f, subfolder='new_results')
for f in context_files:
    _copy(f, subfolder='original_results')
for f in context_figs:
    _copy(f, subfolder='original_results')
for c in copied:
    pass
if missing:
    for m in missing:
        pass
ZIP_PATH = Path('/kaggle/working/robustness_checks.zip') if Path('/kaggle/working').exists() else Path('./robustness_checks.zip')
shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', STAGE_DIR)
try:
    from google.colab import files as _colab_files
    _colab_files.download(str(ZIP_PATH))
except ImportError:
    pass


In [ ]:
import json
from pathlib import Path
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = Path('/kaggle/working/bdhs_outputs')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_cache(filename, kind='json'):
    """Return cached contents if the file exists and is non-empty/valid, else None."""
    path = OUTPUT_DIR / filename
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        if kind == 'json':
            with open(path) as f:
                return json.load(f)
        elif kind == 'csv':
            df = pd.read_csv(path)
            return df if len(df) > 0 else None
    except Exception as e:
        return None
import numpy as np, pandas as pd
import logging
import matplotlib.pyplot as plt
_cache = load_cache('subcluster_bootstrap_results.json')
if _cache is not None:
    subcluster_results = _cache
    for dname, r in subcluster_results.items():
        print(f'  {dname}: full={r['full_division_pct_pos']}% positive, high-drought subset={r['bootstrap_pct_pos']}% positive')
else:
    SEVERITY_TERCILE = 0.667
    B_SUB = 500
    _panel = CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel
    _pathway_divs = {dc: res for dc, res in div_results.items() if dc > 0 and res.get('csi_to_anc4plus', 0) > 0.01}
    subcluster_results = {}
    if not _pathway_divs:
        pass
    else:
        for dc, res in _pathway_divs.items():
            dname = DIVISIONS.get(dc, f'Div{dc}')
            sub_full = _panel[_panel['DIVISION'] == float(dc)].copy()
            if 'CLIMATE_SHOCK_IDX' not in sub_full.columns or len(sub_full) < 20:
                continue
            thresh = sub_full['CLIMATE_SHOCK_IDX'].quantile(SEVERITY_TERCILE)
            sub_high = sub_full[sub_full['CLIMATE_SHOCK_IDX'] >= thresh].copy()
            print(f'  {dname}: full n={len(sub_full)}, high-drought subset n={len(sub_high)} (CSI >= {thresh:.2f})')
            if len(sub_high) < 20:
                continue
            boot_ws = []
            np.random.seed(42)
            for _b in range(B_SUB):
                samp = sub_high.sample(n=len(sub_high), replace=True)
                try:
                    from castle.algorithms import Notears
                    logging.disable(logging.CRITICAL)
                    from sklearn.preprocessing import StandardScaler
                    xs = StandardScaler().fit_transform(samp[CAUSAL_NODE_COLS].dropna())
                    m = Notears(w_threshold=0.01)
                    m.learn(xs)
                    adj = pd.DataFrame(m.causal_matrix, index=CAUSAL_NODE_COLS, columns=CAUSAL_NODE_COLS)
                    logging.disable(logging.NOTSET)
                    ci = CAUSAL_NODE_COLS.index('CLIMATE_SHOCK_IDX')
                    ai = CAUSAL_NODE_COLS.index('ANC_4PLUS')
                    boot_ws.append(float(adj.iloc[ci, ai]))
                except Exception:
                    boot_ws.append(0.0)
            arr = np.array(boot_ws)
            pct = float((arr > 0.01).mean() * 100)
            mean_w, sd_w = (float(arr.mean()), float(arr.std()))
            lo, hi = (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
            print(f'    HIGH-DROUGHT SUBSET: mean={mean_w:.3f} SD={sd_w:.3f} 95%CI=[{lo:.3f},{hi:.3f}] positive={pct:.1f}%')
            subcluster_results[dname] = {'n_full': len(sub_full), 'n_high_drought': len(sub_high), 'csi_threshold': float(thresh), 'bootstrap_mean': round(mean_w, 4), 'bootstrap_sd': round(sd_w, 4), 'bootstrap_95ci_lo': round(lo, 4), 'bootstrap_95ci_hi': round(hi, 4), 'bootstrap_pct_pos': round(pct, 2), 'full_division_pct_pos': conv_log.get(dname, {}).get('bootstrap_pct_pos')}
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.hist(arr, bins=40, color='darkorange', alpha=0.7)
            ax.axvline(mean_w, color='red', lw=2, label=f'Mean={mean_w:.3f}')
            ax.axvline(lo, color='gray', lw=1.2, ls='--', label='2.5/97.5 pct')
            ax.axvline(hi, color='gray', lw=1.2, ls='--')
            ax.set_xlabel('CSI->ANC4+ edge weight')
            ax.set_ylabel('Resamples')
            ax.set_title(f'High-drought subset bootstrap: {dname} (B={B_SUB}, positive={pct:.1f}%)')
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f'fig_bootstrap_subcluster_{dname.lower()}.png', dpi=150, bbox_inches='tight')
            plt.show()
        with open(OUTPUT_DIR / 'subcluster_bootstrap_results.json', 'w') as f:
            json.dump(subcluster_results, f, indent=2)
import copy
_cache = load_cache('ablation_anc_delta_results.json')
if _cache is not None:
    ablation_results = _cache
    for dname, r in ablation_results.items():
        print(f'  {dname}: CAI step original={r['cai_step_original']}, ablation={r['cai_step_ablation']}, preference held={r['preference_held_under_ablation']}')
else:
    ABLATION_TIMESTEPS = TIMESTEPS
    ablation_results = {}
    for div_code in sorted(PATHWAY_DIVISIONS):
        div_name = DIVISIONS.get(div_code, str(div_code))
        print(f'{'-' * 45}\n{div_name} -- ABLATION (anc_delta forced to 0.05)\n{'-' * 45}')
        div_df = MODEL_DF[MODEL_DF['DIVISION'] == div_code].copy()
        if len(div_df) < 30:
            div_df = MODEL_DF.copy()

        class DHSPolicyEnvAblation(DHSPolicyEnv):

            def __init__(self, df, division_code=0, budget=1.0, max_steps=8):
                gym.Env.__init__(self)
                self.df = df.reset_index(drop=True).copy()
                self.division_code = division_code
                self.budget_max = budget
                self.max_steps = max_steps
                self.interventions = copy.deepcopy(INTERVENTIONS_BASE)
                self.state_cols = [c for c in STATE_COLS_CANDIDATES if c in self.df.columns]
                self.observation_space = spaces.Box(low=-5, high=5, shape=(len(self.state_cols) + 1,), dtype=np.float32)
                self.action_space = spaces.Discrete(N_ACTIONS)
                self.scaler = StandardScaler()
                self.df_scaled = pd.DataFrame(self.scaler.fit_transform(self.df[self.state_cols].fillna(0)), columns=self.state_cols)
        vec_env = DummyVecEnv([lambda d=div_df.copy(), dc=div_code: DHSPolicyEnvAblation(d, division_code=dc) for _ in range(N_ENVS)])
        vec_env = VecMonitor(vec_env)
        cb = RewardCallback(ABLATION_TIMESTEPS, f'{div_name}-ablation')
        model = PPO('MlpPolicy', vec_env, seed=42, **PPO_KWARGS)
        model.learn(total_timesteps=ABLATION_TIMESTEPS, callback=cb, progress_bar=False)
        vec_env.close()
        env = DHSPolicyEnvAblation(div_df, division_code=div_code)
        obs, _ = env.reset()
        seq = []
        used = set()
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                feat = model.policy.features_extractor(obs_t)
                lat = model.policy.mlp_extractor(feat)[0]
                logits = model.policy.action_net(lat).cpu().numpy().flatten()
            for u in used:
                logits[u] = -np.inf
            if np.all(logits == -np.inf):
                break
            a = int(np.argmax(logits))
            used.add(a)
            seq.append(env.interventions[a]['name'])
            obs, _, done, _, _ = env.step(a)
        original_seq = rl_results.get(div_code, {}).get('optimal_sequence', [])
        cai_step_ablation = seq.index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in seq else None
        cai_step_original = original_seq.index('Climate Adaptation Infra') + 1 if 'Climate Adaptation Infra' in original_seq else None
        print(f'  Original sequence: {' -> '.join(original_seq[:4])}')
        print(f'  Ablation sequence:  {' -> '.join(seq[:4])}')
        print(f'  CAI step -- original: {cai_step_original}, ablation: {cai_step_ablation}\n')
        ablation_results[div_name] = {'original_sequence': original_seq, 'ablation_sequence': seq, 'cai_step_original': cai_step_original, 'cai_step_ablation': cai_step_ablation, 'preference_held_under_ablation': cai_step_ablation == cai_step_original}
    with open(OUTPUT_DIR / 'ablation_anc_delta_results.json', 'w') as f:
        json.dump(ablation_results, f, indent=2)
_cache = load_cache('scale_invariance_check.json')
if _cache is not None:
    rescale_results = _cache
    for dname, r in rescale_results.items():
        print(f'  {dname}: {r['n_present_of_total']} rescales positive, scale-stable={r['scale_stable']}, reversal={r['any_edge_reversal']}')
else:
    from sklearn.preprocessing import StandardScaler
    RESCALE_FACTORS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
    _panel = CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel
    _pathway_divs = {dc: res for dc, res in div_results.items() if dc > 0 and res.get('csi_to_anc4plus', 0) > 0.01}
    rescale_results = {}
    if not _pathway_divs:
        pass
    else:
        for dc, res in _pathway_divs.items():
            dname = DIVISIONS.get(dc, f'Div{dc}')
            sub = _panel[_panel['DIVISION'] == float(dc)].copy()
            if 'CLIMATE_SHOCK_IDX' not in sub.columns or len(sub) < 20:
                continue
            print(f'  {dname} (n={len(sub)}):')
            division_results = []
            for factor in RESCALE_FACTORS:
                sub_rescaled = sub.copy()
                sub_rescaled['CLIMATE_SHOCK_IDX'] = sub_rescaled['CLIMATE_SHOCK_IDX'] * factor
                try:
                    from castle.algorithms import Notears
                    logging.disable(logging.CRITICAL)
                    xs = StandardScaler().fit_transform(sub_rescaled[CAUSAL_NODE_COLS].dropna())
                    m = Notears(w_threshold=0.01)
                    m.learn(xs)
                    adj = pd.DataFrame(m.causal_matrix, index=CAUSAL_NODE_COLS, columns=CAUSAL_NODE_COLS)
                    logging.disable(logging.NOTSET)
                    ci = CAUSAL_NODE_COLS.index('CLIMATE_SHOCK_IDX')
                    ai = CAUSAL_NODE_COLS.index('ANC_4PLUS')
                    edge_w = float(adj.iloc[ci, ai])
                    edge_w_reversed = float(adj.iloc[ai, ci])
                except Exception:
                    edge_w, edge_w_reversed = (0.0, 0.0)
                present = edge_w > 0.01
                reversed_present = edge_w_reversed > 0.01
                print(f'    factor={factor:>5.1f}x  CSI->ANC4+ w={edge_w:.3f}  ({('present' if present else 'absent')}){('  [REVERSED]' if reversed_present else '')}')
                division_results.append({'rescale_factor': factor, 'edge_weight_forward': round(edge_w, 4), 'edge_present': bool(present), 'edge_weight_reversed': round(edge_w_reversed, 4), 'reversed_edge_present': bool(reversed_present)})
            n_present = sum((r['edge_present'] for r in division_results))
            n_total = len(division_results)
            any_reversed = any((r['reversed_edge_present'] for r in division_results))
            stable = n_present == n_total or n_present == 0
            print(f'    -> {n_present}/{n_total} rescales positive ({('STABLE' if stable else 'UNSTABLE')})\n')
            rescale_results[dname] = {'n_obs': len(sub), 'results_by_factor': division_results, 'n_present_of_total': f'{n_present}/{n_total}', 'scale_stable': stable, 'any_edge_reversal': any_reversed}
        with open(OUTPUT_DIR / 'scale_invariance_check.json', 'w') as f:
            json.dump(rescale_results, f, indent=2)
_cache = load_cache('generalized_edge_stability.json')
if _cache is not None:
    n_total = len(_cache)
    n_unstable = sum((1 for r in _cache if r.get('scale_stable') is False))
    print(f'  {n_unstable}/{n_total} ({100 * n_unstable / n_total:.1f}%) scale-unstable' if n_total else '  No edges in cache.')
    edge_stability_results = _cache
else:
    from sklearn.preprocessing import StandardScaler
    RESCALE_FACTORS_E = [0.2, 1.0, 5.0]
    _panel = CLUSTER_PANEL_IMPUTED if 'CLUSTER_PANEL_IMPUTED' in dir() else cluster_panel

    def _fit_notears(df_sub, node_cols):
        try:
            from castle.algorithms import Notears
            logging.disable(logging.CRITICAL)
            xs = StandardScaler().fit_transform(df_sub[node_cols].dropna())
            m = Notears(w_threshold=0.01)
            m.learn(xs)
            adj = pd.DataFrame(m.causal_matrix, index=node_cols, columns=node_cols)
            logging.disable(logging.NOTSET)
            return adj
        except Exception:
            return None
    all_candidate_edges = []
    for dc, res in div_results.items():
        if dc == 0:
            continue
        adj = res.get('adj')
        if adj is None:
            continue
        for src in CAUSAL_NODE_COLS:
            for tgt in CAUSAL_NODE_COLS:
                if src == tgt or src not in adj.index or tgt not in adj.columns:
                    continue
                w = float(adj.loc[src, tgt])
                if w > 0.01:
                    all_candidate_edges.append((dc, src, tgt, w))
    tested_pairs = sorted(set(((e[0], e[1]) for e in all_candidate_edges)))
    edge_stability_results = []
    for i, (dc, src_node) in enumerate(tested_pairs, 1):
        dname = DIVISIONS.get(dc, f'Div{dc}')
        sub = _panel[_panel['DIVISION'] == float(dc)].copy()
        if len(sub) < 20:
            continue
        print(f'  [{i}/{len(tested_pairs)}] {dname} -- rescaling {src_node}...')
        orig_edges = [(e[2], e[3]) for e in all_candidate_edges if e[0] == dc and e[1] == src_node]
        factor_adjs = {}
        for factor in RESCALE_FACTORS_E:
            sub_rescaled = sub.copy()
            sub_rescaled[src_node] = sub_rescaled[src_node] * factor
            factor_adjs[factor] = _fit_notears(sub_rescaled, CAUSAL_NODE_COLS)
        for tgt_node, orig_w in orig_edges:
            weights_by_factor = {}
            for factor, adj in factor_adjs.items():
                if adj is None:
                    weights_by_factor[factor] = None
                    continue
                w = float(adj.loc[src_node, tgt_node]) if src_node in adj.index and tgt_node in adj.columns else 0.0
                weights_by_factor[factor] = w
            valid_weights = [w for w in weights_by_factor.values() if w is not None]
            present_flags = [w > 0.01 for w in valid_weights]
            stable = len(set(present_flags)) <= 1 if present_flags else None
            edge_stability_results.append({'division': dname, 'source': src_node, 'target': tgt_node, 'original_weight': orig_w, 'weights_by_rescale_factor': weights_by_factor, 'scale_stable': stable})
    n_total = len(edge_stability_results)
    n_stable = sum((1 for r in edge_stability_results if r['scale_stable']))
    n_unstable = sum((1 for r in edge_stability_results if r['scale_stable'] is False))
    print(f'Total edges tested: {n_total} | Stable: {n_stable} | Unstable: {n_unstable}')
    if n_total:
        print(f'Pct unstable: {100 * n_unstable / n_total:.1f}%')
    with open(OUTPUT_DIR / 'generalized_edge_stability.json', 'w') as f:
        json.dump(edge_stability_results, f, indent=2, default=str)
    summary_df = pd.DataFrame([{'Total edges tested': n_total, 'Scale-stable': n_stable, 'Scale-unstable': n_unstable, 'Pct unstable': round(100 * n_unstable / n_total, 1) if n_total else None}])
    summary_df.to_csv(OUTPUT_DIR / 'table_generalized_edge_stability.csv', index=False)


In [ ]:
import shutil, json
from pathlib import Path
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = Path('/kaggle/working/bdhs_outputs')
WORK_DIR = Path('/kaggle/working')
STAGE = WORK_DIR / 'paper_files'
shutil.rmtree(STAGE, ignore_errors=True)
PACKAGE = {'figures/main': {'fig1_stunting_trend.png': 'Fig1A_stunting_trend.png', 'fig2_csi.png': 'Fig1B_climate_shock_index.png', 'fig3_dag.png': 'Fig2A_causal_dag.png', 'fig4_vae_clusters.png': 'Fig3A_vae_clusters.png', 'fig5_ctgan.png': 'Fig3B_ctgan_fidelity.png', 'fig6_shap.png': 'Fig4_shap_importance.png', 'fig7_pareto.png': 'Fig5A_pareto_frontier.png'}, 'figures/robustness': {'fig_bootstrap_rajshahi.png': 'Fig2B_bootstrap_rajshahi_full.png', 'fig_bootstrap_khulna.png': 'Fig2C_bootstrap_khulna_full.png', 'fig_bootstrap_subcluster_rajshahi.png': 'Fig2D_bootstrap_rajshahi_subcluster.png', 'fig_bootstrap_subcluster_khulna.png': 'Fig2E_bootstrap_khulna_subcluster.png'}, 'tables': {'table1_linkage.csv': 'Table1_linkage.csv', 'table2_causal_discovery.csv': 'Table2_causal_discovery.csv', 'table3_rl_sequences.csv': 'Table3_rl_sequences.csv', 'table_ablation_anc_delta.csv': 'TableS3_ablation_summary.csv', 'table_generalized_edge_stability.csv': 'TableS4_generalized_edge_stability.csv', 'shap_feature_importance.csv': 'TableS5_shap_importance.csv', 'synthetic_fidelity.csv': 'TableS6_ctgan_fidelity.csv', 'vae_cluster_profiles.csv': 'TableS7_vae_profiles.csv', 'pareto_frontier.csv': 'TableS8_pareto_frontier.csv'}, 'data/robustness': {'notears_convergence_log.json': 'notears_convergence_log.json', 'subcluster_bootstrap_results.json': 'subcluster_bootstrap_results.json', 'ablation_anc_delta_results.json': 'ablation_anc_delta_results.json', 'scale_invariance_check.json': 'scale_invariance_check.json', 'generalized_edge_stability.json': 'generalized_edge_stability.json'}, 'reviewer_additions': {'table_zscore_vs_raw_notears.csv': 'TableR1_zscore_vs_raw.csv', 'table_lambda_grid_stability.csv': 'TableR2_lambda_grid_stability.csv', 'table_negative_controls.csv': 'TableR3_negative_controls.csv', 'table_cross_algorithm_consensus.csv': 'TableR4_cross_algorithm_consensus.csv', 'table_sensitivity_analysis.csv': 'TableR5_sensitivity_ranges.csv', 'table_ppo_multirun_uncertainty.csv': 'TableR6_ppo_uncertainty.csv', 'table_stronger_ablation_no_csi.csv': 'TableR7_stronger_ablation.csv', 'table_greedy_sequences.csv': 'TableR8_greedy_sequences.csv', 'zscore_vs_raw_notears.json': 'zscore_vs_raw.json', 'lambda_grid_stability.json': 'lambda_grid.json', 'negative_control_results.json': 'negative_controls.json', 'cross_algorithm_consensus.json': 'cross_algorithm.json', 'tvae_vs_ctgan.json': 'tvae_vs_ctgan.json', 'synthetic_detectability.json': 'detectability.json', 'baseline_comparison.json': 'baseline_comparison.json', 'ppo_multirun_uncertainty.json': 'ppo_uncertainty.json', 'stronger_ablation_no_csi.json': 'stronger_ablation.json', 'sensitivity_analysis.json': 'sensitivity.json'}, 'policy_briefs': {'policy_brief_barisal.txt': 'policy_brief_barisal.txt', 'policy_brief_chittagong.txt': 'policy_brief_chittagong.txt', 'policy_brief_dhaka.txt': 'policy_brief_dhaka.txt', 'policy_brief_khulna.txt': 'policy_brief_khulna.txt', 'policy_brief_mymensingh.txt': 'policy_brief_mymensingh.txt', 'policy_brief_rajshahi.txt': 'policy_brief_rajshahi.txt', 'policy_brief_rangpur.txt': 'policy_brief_rangpur.txt', 'policy_brief_sylhet.txt': 'policy_brief_sylhet.txt'}}
copied, missing = ([], [])
for subfolder, file_map in PACKAGE.items():
    dest_dir = STAGE / subfolder
    dest_dir.mkdir(parents=True, exist_ok=True)
    for src_name, dest_name in file_map.items():
        src = OUTPUT_DIR / src_name
        if src.exists() and src.stat().st_size > 0:
            shutil.copy2(src, dest_dir / dest_name)
            copied.append(f'{subfolder}/{dest_name}')
        else:
            missing.append(src_name)
readme = STAGE / 'README.txt'
readme.write_text(f'BDHS GENERATIVE CAUSAL AI — PAPER WRITING FILES\n=================================================\nGenerated from: lancet-casual-ai notebook (full run)\nPackaged: {len(copied)} files\n\nFOLDER STRUCTURE:\n  figures/main/        7 main pipeline figures (Fig1–Fig5)\n  figures/robustness/  4 bootstrap histogram figures (Fig2 panels B-E)\n  tables/              3 main tables + 6 supplementary tables (CSV)\n  data/robustness/     5 JSON files with exact numbers for verification\n  policy_briefs/       8 division-level policy brief text files\n\nKEY NUMBERS FOR MANUSCRIPT:\n  Clusters linked (all 4 waves): 596 / 675 (88.3%)\n  Mean SHD across divisions: 52.9\n  Pathway divisions: Khulna, Rajshahi (CSI->ANC4+ edge = 1.000)\n  Bootstrap stability (full division):\n    Khulna:   77.6% positive (95% CI 0.000-1.000)\n    Rajshahi: 41.6% positive (95% CI 0.000-1.000)\n  Bootstrap stability (high-drought sub-cluster):\n    Khulna:   23.4% positive  [DECREASED — pathway not concentrated]\n    Rajshahi: 27.6% positive  [DECREASED — pathway not concentrated]\n  Ablation (CAI step, original vs ablated):\n    Khulna:   step 1 -> step 4  [preference NOT preserved]\n    Rajshahi: step 8 -> step 6  [preference NOT preserved]\n  Scale-stability (Cell D, CSI->ANC4+ edge, 6 factors):\n    Khulna:   6/6 stable  [NOT a Kaiser-Sipos scale artifact]\n    Rajshahi: 6/6 stable  [NOT a Kaiser-Sipos scale artifact]\n  Generalized edge stability (Cell E, all edges):\n    265/265 edges scale-stable (0.0% unstable)\n  VAE: silhouette=0.271, k=3\n  CTGAN: W1=0.2135, PCMAD=0.0523\n  LightGBM AUC: 0.6435, top feature: WEALTH_Q\n  PPO: random=0.6411, uniform=1.4180, trained=1.4829\n  Pareto knee: H=0.3938, Gini=0.0538 (5 runs stable)\n\nMISSING FILES (not found in OUTPUT_DIR):\n' + ('\n'.join((f'  - {m}' for m in missing)) if missing else '  None — all files found.'))
ZIP_PATH = WORK_DIR / 'paper_writing_files.zip'
shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', STAGE)
size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f'  PAPER WRITING PACKAGE READY')
if missing:
    for m in missing:
        print(f'    - {m}')
print(f'  Location:      {ZIP_PATH}')
try:
    from google.colab import files as _cf
    _cf.download(str(ZIP_PATH))
except ImportError:
    pass
